In [1]:
import pandas as pd
import numpy as np
from itertools import product
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
DATA_DIR      = './data'
TEMPLATE_PATH = '/home/sagemaker-user/data/template_forecast_v00.csv'
OUTPUT_PATH   = '/home/sagemaker-user/data/August_2026_Forecast_v01.csv'
PORTFOLIOS    = ['A', 'B', 'C', 'D']
MONTH_MAP     = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
                 'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD & CLEAN INTERVAL DATA (Apr-Jun 2025)
# ══════════════════════════════════════════════════════════════════════════════
print("Step 1: Loading interval data...")

def load_interval(portfolio):
    df = pd.read_csv(f'{DATA_DIR}/{portfolio} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()

    # Build complete skeleton — every 30-min slot Apr 1 to Jun 30
    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })

    df = skeleton.merge(df, on=['Month','Day','Interval'], how='left')

    # Interpolate numeric columns within each day
    for col in ['Call Volume','CCT','Abandoned Calls','Abandoned Rate','Service Level']:
        if col in df.columns:
            df[col] = (
                df.groupby(['Month','Day'])[col]
                  .transform(lambda x: x.interpolate(method='linear').bfill().ffill())
            )
            # Fallback: same DOW + interval median
            df['_dow'] = pd.to_datetime(
                df['Month'] + ' ' + df['Day'].astype(str) + ' 2025'
            ).dt.dayofweek
            df['_iod'] = df['Interval'].apply(
                lambda x: int(x.split(':')[0])*2 + int(x.split(':')[1])//30
            )
            df[col] = df.groupby(['_dow','_iod'])[col].transform(
                lambda x: x.fillna(x.median())
            )

    # Clip negatives
    for col in ['Call Volume','CCT','Abandoned Calls','Abandoned Rate']:
        if col in df.columns:
            df[col] = df[col].clip(lower=0)
    if 'Abandoned Rate' in df.columns:
        df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    # Final time features
    df['Date']       = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['day_of_month'] = df['Date'].dt.day
    df['month']        = df['Date'].dt.month
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday']   = df['Date'].isin(
        pd.to_datetime(['2025-05-26','2025-06-19'])
    ).astype(int)
    df['IntervalIdx']  = df['_iod']
    df['hour_sin'] = np.sin(2*np.pi*df['IntervalIdx']/48)
    df['hour_cos'] = np.cos(2*np.pi*df['IntervalIdx']/48)
    df['dow_sin']  = np.sin(2*np.pi*df['day_of_week']/7)
    df['dow_cos']  = np.cos(2*np.pi*df['day_of_week']/7)
    df['Portfolio'] = portfolio
    df = df.drop(columns=['_dow','_iod'], errors='ignore')

    return df

interval_dfs = {}
for p in PORTFOLIOS:
    interval_dfs[p] = load_interval(p)
    print(f"  {p}: {interval_dfs[p].shape}, nulls: {interval_dfs[p][['Call Volume','CCT','Abandoned Rate']].isnull().sum().sum()}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — LOAD & CLEAN DAILY DATA (Jan 2024 - Dec 2025)
# ══════════════════════════════════════════════════════════════════════════════
print("\nStep 2: Loading daily data...")

def load_daily(portfolio):
    df = pd.read_csv(f'{DATA_DIR}/{portfolio} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Date']         = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df                 = df.sort_values('Date').reset_index(drop=True)
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    # Fill missing with same-DOW median
    for col in ['Call Volume','CCT','Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(
            lambda x: x.fillna(x.median())
        )
    return df

daily_dfs = {}
for p in PORTFOLIOS:
    daily_dfs[p] = load_daily(p)
    print(f"  {p}: {daily_dfs[p].shape}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — MERGE DAILY TOTALS INTO INTERVAL DATA
# ══════════════════════════════════════════════════════════════════════════════
print("\nStep 3: Merging daily totals into interval data...")

for p in PORTFOLIOS:
    interval_dfs[p] = interval_dfs[p].merge(
        daily_dfs[p][['Date','Call Volume','CCT','Abandon Rate']].rename(columns={
            'Call Volume': 'daily_cv',
            'CCT':         'daily_cct',
            'Abandon Rate':'daily_abd'
        }),
        on='Date', how='left'
    )

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — STAGE 1: TRAIN DAILY MODELS & PREDICT AUGUST 2026 DAILY TOTALS
# ══════════════════════════════════════════════════════════════════════════════
print("\nStep 4: Training daily models...")

DAILY_FEATURES = ['day_of_week','month','day_of_month','week_of_month',
                  'is_weekend','dow_sin','dow_cos','month_sin','month_cos',
                  'cv_lag_364','cct_lag_364','abd_lag_364']

aug_days = pd.date_range('2026-08-01','2026-08-31', freq='D')
all_aug_daily = []

for p in PORTFOLIOS:
    d = daily_dfs[p].copy()
    d['dow_sin']   = np.sin(2*np.pi*d['day_of_week']/7)
    d['dow_cos']   = np.cos(2*np.pi*d['day_of_week']/7)
    d['month_sin'] = np.sin(2*np.pi*d['month']/12)
    d['month_cos'] = np.cos(2*np.pi*d['month']/12)
    d['cv_lag_364']  = d['Call Volume'].shift(364)
    d['cct_lag_364'] = d['CCT'].shift(364)
    d['abd_lag_364'] = d['Abandon Rate'].shift(364)

    train = d.dropna(subset=DAILY_FEATURES + ['Call Volume'])

    # Train 3 daily models
    daily_models = {}
    for target in ['Call Volume','CCT','Abandon Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, max_depth=5, random_state=42
        )
        m.fit(train[DAILY_FEATURES], train[target])
        daily_models[target] = m

    # Build August 2026 grid
    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'month':         aug_days.month,
        'day_of_month':  aug_days.day,
        'week_of_month': (aug_days.day - 1) // 7 + 1,
        'is_weekend':    (aug_days.dayofweek >= 5).astype(int),
    })
    aug['dow_sin']   = np.sin(2*np.pi*aug['day_of_week']/7)
    aug['dow_cos']   = np.cos(2*np.pi*aug['day_of_week']/7)
    aug['month_sin'] = np.sin(2*np.pi*aug['month']/12)
    aug['month_cos'] = np.cos(2*np.pi*aug['month']/12)

    # Lag = August 2025
    aug_2025 = d[
        (d['Date'].dt.month == 8) & (d['Date'].dt.year == 2025)
    ].reset_index(drop=True)
    aug['cv_lag_364']  = aug_2025['Call Volume'].values
    aug['cct_lag_364'] = aug_2025['CCT'].values
    aug['abd_lag_364'] = aug_2025['Abandon Rate'].values

    for target, col in [('Call Volume','pred_cv'),('CCT','pred_cct'),('Abandon Rate','pred_abd')]:
        aug[col] = np.maximum(0, daily_models[target].predict(aug[DAILY_FEATURES]))

    aug['Portfolio'] = p
    all_aug_daily.append(aug)
    print(f"  {p}: daily models trained on {len(train)} rows")

aug_daily_all = pd.concat(all_aug_daily, ignore_index=True)
print(f"  August daily grid: {aug_daily_all.shape}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — STAGE 2: TRAIN INTERVAL MODELS & PREDICT AUGUST INTERVALS
# ══════════════════════════════════════════════════════════════════════════════
print("\nStep 5: Training interval models...")

INTERVAL_FEATURES = [
    'IntervalIdx','day_of_week','day_of_month','month',
    'week_of_month','is_weekend','is_holiday',
    'hour_sin','hour_cos','dow_sin','dow_cos',
    'daily_cv'  # Stage 1 output as context feature
]

all_forecasts = []

for p in PORTFOLIOS:
    df = interval_dfs[p].copy()

    # Train on April + May, validate on June
    train_iv = df[df['month'].isin([4,5])].dropna(subset=INTERVAL_FEATURES)
    val_iv   = df[df['month'] == 6].dropna(subset=INTERVAL_FEATURES)

    iv_models = {}
    for target in ['Call Volume','CCT','Abandoned Rate']:
        tr = train_iv.dropna(subset=[target])
        vl = val_iv.dropna(subset=[target])

        m = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05,
            max_depth=6, min_samples_leaf=20,
            random_state=42
        )
        m.fit(tr[INTERVAL_FEATURES], tr[target])

        preds = np.maximum(0, m.predict(vl[INTERVAL_FEATURES]))
        mask  = vl[target] > 0
        if mask.sum() > 0:
            mape = mean_absolute_percentage_error(
                vl[target][mask], preds[mask]
            ) * 100
            print(f"  {p} {target}: MAPE = {mape:.2f}%")

        iv_models[target] = m

    # Generate August 2026 interval forecasts
    aug_p = aug_daily_all[aug_daily_all['Portfolio'] == p]

    rows = []
    for _, day_row in aug_p.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])
        wom = int((dom - 1) // 7 + 1)

        for iod in range(48):
            h = iod // 2
            m = (iod % 2) * 30

            feat = pd.DataFrame([{
                'IntervalIdx':   iod,
                'day_of_week':   dow,
                'day_of_month':  dom,
                'month':         8,
                'week_of_month': wom,
                'is_weekend':    int(dow >= 5),
                'is_holiday':    0,
                'hour_sin':      np.sin(2*np.pi*iod/48),
                'hour_cos':      np.cos(2*np.pi*iod/48),
                'dow_sin':       np.sin(2*np.pi*dow/7),
                'dow_cos':       np.cos(2*np.pi*dow/7),
                'daily_cv':      day_row['pred_cv'],
            }])

            cv_pred  = max(0, iv_models['Call Volume'].predict(feat)[0])
            cct_pred = max(0, iv_models['CCT'].predict(feat)[0])
            abd_pred = max(0, min(1, iv_models['Abandoned Rate'].predict(feat)[0]))

            rows.append({
                'Month':           'August',
                'Day':             dom,
                'Interval':        f"{h}:{m:02d}",
                'Portfolio':       p,
                f'Calls_Offered_{p}':   round(cv_pred, 2),
                f'Abandoned_Calls_{p}': round(cv_pred * abd_pred, 2),
                f'Abandoned_Rate_{p}':  round(abd_pred, 6),
                f'CCT_{p}':             round(cct_pred, 2),
            })

    all_forecasts.append(pd.DataFrame(rows))
    print(f"  {p}: {len(rows)} interval forecasts generated")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — COMBINE ALL PORTFOLIOS INTO TEMPLATE FORMAT
# ══════════════════════════════════════════════════════════════════════════════
print("\nStep 6: Building final CSV...")

# Start with A as base
result = all_forecasts[0][['Month','Day','Interval',
    'Calls_Offered_A','Abandoned_Calls_A','Abandoned_Rate_A','CCT_A']]

# Merge B, C, D on Month+Day+Interval
for i, p in enumerate(['B','C','D']):
    merge_cols = ['Month','Day','Interval',
        f'Calls_Offered_{p}',f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}',f'CCT_{p}']
    result = result.merge(
        all_forecasts[i+1][merge_cols],
        on=['Month','Day','Interval'], how='left'
    )

# Enforce column order to match template
col_order = ['Month','Day','Interval']
for p in PORTFOLIOS:
    col_order += [f'Calls_Offered_{p}',f'Abandoned_Calls_{p}',
                  f'Abandoned_Rate_{p}',f'CCT_{p}']
result = result[col_order]

# Final clip — no negatives
for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

# Save
result.to_csv(OUTPUT_PATH, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n✅ Saved to {OUTPUT_PATH}")
print(f"   Shape: {result.shape} | Expected: (1488, 19)")
print(f"   Any negatives: {(result.select_dtypes('number') < 0).any().any()}")
print(f"   Any nulls: {result.isnull().any().any()}")

print("\n=== Daily CV totals by portfolio ===")
for p in PORTFOLIOS:
    col = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")

print("\n=== Sample output (Day 1) ===")
print(result[result['Day']==1].head(5).to_string())

Step 1: Loading interval data...
  A: (4368, 21), nulls: 0
  B: (4368, 21), nulls: 0
  C: (4368, 21), nulls: 0
  D: (4368, 21), nulls: 0

Step 2: Loading daily data...
  A: (731, 10)
  B: (731, 10)
  C: (731, 10)
  D: (731, 10)

Step 3: Merging daily totals into interval data...

Step 4: Training daily models...
  A: daily models trained on 367 rows
  B: daily models trained on 367 rows
  C: daily models trained on 367 rows
  D: daily models trained on 367 rows
  August daily grid: (124, 17)

Step 5: Training interval models...
  A Call Volume: MAPE = 32.66%
  A CCT: MAPE = 31.62%
  A Abandoned Rate: MAPE = 66.80%
  A: 1488 interval forecasts generated
  B Call Volume: MAPE = 28.46%
  B CCT: MAPE = 21.65%
  B Abandoned Rate: MAPE = 137.92%
  B: 1488 interval forecasts generated
  C Call Volume: MAPE = 12.44%
  C CCT: MAPE = 9.75%
  C Abandoned Rate: MAPE = 204.39%
  C: 1488 interval forecasts generated
  D Call Volume: MAPE = 18.59%
  D CCT: MAPE = 19.35%
  D Abandoned Rate: MAPE = 144

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — update paths to match your SageMaker setup
# ══════════════════════════════════════════════════════════════════════════════
DATA_DIR    = './data'
OUTPUT_PATH = './forecast_v01.csv'
PORTFOLIOS  = ['A', 'B', 'C', 'D']

DAILY_FEATURES = [
    'day_of_week', 'month', 'day_of_month', 'week_of_month', 'is_weekend',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'cv_lag_364', 'cct_lag_364', 'abd_lag_364'
]
INTERVAL_FEATURES = [
    'IntervalIdx', 'day_of_week', 'day_of_month', 'month', 'week_of_month',
    'is_weekend', 'is_holiday', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'daily_cv'
]

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN INTERVAL DATA (Apr-Jun 2025)
# ══════════════════════════════════════════════════════════════════════════════
def load_interval(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()

    # Convert time objects to string if needed
    df['Interval'] = df['Interval'].apply(
        lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x)
    )

    # Build complete skeleton — every 30-min slot Apr 1 to Jun 30
    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })
    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    # Interpolate within each day then clip
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = (
            df.groupby(['Month', 'Day'])[col]
              .transform(lambda x: x.interpolate(method='linear').bfill().ffill())
              .clip(lower=0)
        )
    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    # Time features
    df['Date']         = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['day_of_month'] = df['Date'].dt.day
    df['month']        = df['Date'].dt.month
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday']   = df['Date'].isin(
        pd.to_datetime(['2025-05-26', '2025-06-19'])
    ).astype(int)
    df['IntervalIdx']  = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )
    df['hour_sin'] = np.sin(2 * np.pi * df['IntervalIdx'] / 48)
    df['hour_cos'] = np.cos(2 * np.pi * df['IntervalIdx'] / 48)
    df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)
    return df


# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN DAILY DATA (Jan 2024 - Dec 2025)
# ══════════════════════════════════════════════════════════════════════════════
def load_daily(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Date']         = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df                 = df.sort_values('Date').reset_index(drop=True)
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    # Fill missing with same-DOW median
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(
            lambda x: x.fillna(x.median())
        )

    # Cyclical + lag features
    df['dow_sin']    = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']    = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)
    df['cv_lag_364']  = df['Call Volume'].shift(364)   # same DOW last year
    df['cct_lag_364'] = df['CCT'].shift(364)
    df['abd_lag_364'] = df['Abandon Rate'].shift(364)
    return df


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
aug_days      = pd.date_range('2026-08-01', '2026-08-31', freq='D')
all_forecasts = []

for p in PORTFOLIOS:
    print(f"\n{'='*50}")
    print(f"Portfolio {p}")
    print(f"{'='*50}")

    d  = load_daily(p)
    iv = load_interval(p)

    # Merge daily totals into interval data as context feature
    iv = iv.merge(
        d[['Date', 'Call Volume']].rename(columns={'Call Volume': 'daily_cv'}),
        on='Date', how='left'
    )

    # ── STAGE 1: Train daily models ───────────────────────────────────────────
    train_d = d.dropna(subset=DAILY_FEATURES + ['Call Volume'])
    print(f"Daily training rows: {len(train_d)}")

    daily_models = {}
    for target in ['Call Volume', 'CCT', 'Abandon Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.05, max_depth=5, random_state=42
        )
        m.fit(train_d[DAILY_FEATURES], train_d[target])
        daily_models[target] = m

    # Build August 2026 daily grid
    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'month':         aug_days.month,
        'day_of_month':  aug_days.day,
        'week_of_month': (aug_days.day - 1) // 7 + 1,
        'is_weekend':    (aug_days.dayofweek >= 5).astype(int),
    })
    aug['dow_sin']   = np.sin(2 * np.pi * aug['day_of_week'] / 7)
    aug['dow_cos']   = np.cos(2 * np.pi * aug['day_of_week'] / 7)
    aug['month_sin'] = np.sin(2 * np.pi * aug['month'] / 12)
    aug['month_cos'] = np.cos(2 * np.pi * aug['month'] / 12)

    # Lag = same DOW from August 2025
    aug_2025 = d[
        (d['Date'].dt.month == 8) & (d['Date'].dt.year == 2025)
    ].reset_index(drop=True)
    aug['cv_lag_364']  = aug_2025['Call Volume'].values
    aug['cct_lag_364'] = aug_2025['CCT'].values
    aug['abd_lag_364'] = aug_2025['Abandon Rate'].values

    for target, col in [
        ('Call Volume', 'pred_cv'),
        ('CCT',         'pred_cct'),
        ('Abandon Rate','pred_abd')
    ]:
        aug[col] = np.maximum(0, daily_models[target].predict(aug[DAILY_FEATURES]))

    print("Stage 1 daily predictions:")
    print(aug[['Date', 'pred_cv', 'pred_cct', 'pred_abd']].to_string())

    # ── STAGE 2: Train interval models ───────────────────────────────────────
    train_iv = iv[iv['month'].isin([4, 5])].dropna(
        subset=INTERVAL_FEATURES + ['Call Volume', 'CCT', 'Abandoned Rate']
    )
    val_iv = iv[iv['month'] == 6].dropna(subset=INTERVAL_FEATURES + ['Call Volume'])
    print(f"\nInterval training rows: {len(train_iv)} | Val rows: {len(val_iv)}")

    iv_models = {}
    for target in ['Call Volume', 'CCT', 'Abandoned Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.05,
            max_depth=6, min_samples_leaf=20, random_state=42
        )
        m.fit(train_iv[INTERVAL_FEATURES], train_iv[target])

        preds = np.maximum(0, m.predict(val_iv[INTERVAL_FEATURES]))
        mask  = val_iv[target] > 0
        if mask.sum() > 0:
            mape = mean_absolute_percentage_error(
                val_iv[target][mask], preds[mask]
            ) * 100
            print(f"  {target}: Val MAPE = {mape:.2f}%")

        iv_models[target] = m

    # ── Generate August interval forecasts (vectorised) ───────────────────────
    aug_grid_rows = []
    for _, r in aug.iterrows():
        for iod in range(48):
            aug_grid_rows.append({
                'Day':           int(r.day_of_month),
                'day_of_week':   int(r.day_of_week),
                'day_of_month':  int(r.day_of_month),
                'week_of_month': int((r.day_of_month - 1) // 7 + 1),
                'is_weekend':    int(r.day_of_week >= 5),
                'pred_cv':       r.pred_cv,
                'IntervalIdx':   iod,
            })

    aug_grid = pd.DataFrame(aug_grid_rows)
    aug_grid['month']      = 8
    aug_grid['is_holiday'] = 0
    aug_grid['hour_sin']   = np.sin(2 * np.pi * aug_grid['IntervalIdx'] / 48)
    aug_grid['hour_cos']   = np.cos(2 * np.pi * aug_grid['IntervalIdx'] / 48)
    aug_grid['dow_sin']    = np.sin(2 * np.pi * aug_grid['day_of_week'] / 7)
    aug_grid['dow_cos']    = np.cos(2 * np.pi * aug_grid['day_of_week'] / 7)
    aug_grid['daily_cv']   = aug_grid['pred_cv']

    aug_grid[f'Calls_Offered_{p}']   = np.maximum(
        0, iv_models['Call Volume'].predict(aug_grid[INTERVAL_FEATURES])
    ).round(2)
    aug_grid[f'CCT_{p}']             = np.maximum(
        0, iv_models['CCT'].predict(aug_grid[INTERVAL_FEATURES])
    ).round(2)
    aug_grid[f'Abandoned_Rate_{p}']  = np.clip(
        iv_models['Abandoned Rate'].predict(aug_grid[INTERVAL_FEATURES]), 0, 1
    ).round(6)
    aug_grid[f'Abandoned_Calls_{p}'] = (
        aug_grid[f'Calls_Offered_{p}'] * aug_grid[f'Abandoned_Rate_{p}']
    ).round(2)

    aug_grid['Month']    = 'August'
    aug_grid['Interval'] = aug_grid['IntervalIdx'].apply(
        lambda x: f"{x // 2}:{(x % 2) * 30:02d}"
    )

    all_forecasts.append(aug_grid[[
        'Month', 'Day', 'Interval',
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]])
    print(f"  {p}: {len(aug_grid)} interval rows generated")


# ══════════════════════════════════════════════════════════════════════════════
# COMBINE ALL PORTFOLIOS INTO FINAL CSV
# ══════════════════════════════════════════════════════════════════════════════
print("\nCombining portfolios...")
result = all_forecasts[0]
for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(
        all_forecasts[i + 1],
        on=['Month', 'Day', 'Interval'], how='left'
    )

# Enforce exact column order
col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]
result = result[col_order]

# Final safety clip
for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n✅ Saved to {OUTPUT_PATH}")
print(f"   Shape:     {result.shape}  |  Expected: (1488, 19)")
print(f"   Negatives: {(result.select_dtypes('number') < 0).any().any()}")
print(f"   Nulls:     {result.isnull().any().any()}")

print("\n=== Daily CV totals by portfolio ===")
for p in PORTFOLIOS:
    col   = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")

print("\n=== First 5 rows ===")
print(result.head(5).to_string())


Portfolio A
Daily training rows: 367
Stage 1 daily predictions:
         Date      pred_cv    pred_cct  pred_abd
0  2026-08-01  3688.237510  311.035536  0.012869
1  2026-08-02  2500.127818  324.546123  0.013132
2  2026-08-03  3057.401255  316.429409  0.012072
3  2026-08-04  4439.593552  314.238150  0.014718
4  2026-08-05  4161.053179  317.976280  0.012699
5  2026-08-06  4256.558946  320.245823  0.013548
6  2026-08-07  4353.007490  330.433416  0.009236
7  2026-08-08  3660.767881  331.719743  0.011548
8  2026-08-09  2186.271467  325.870657  0.012079
9  2026-08-10  2831.888791  319.880048  0.011304
10 2026-08-11  4267.114513  320.385564  0.012979
11 2026-08-12  4047.827072  332.156060  0.012216
12 2026-08-13  4014.519442  317.331218  0.016875
13 2026-08-14  3974.072158  319.510171  0.016034
14 2026-08-15  3329.840280  334.366844  0.009234
15 2026-08-16  2079.776055  322.541435  0.010072
16 2026-08-17  2781.543304  319.875157  0.014277
17 2026-08-18  4132.713429  324.147122  0.009751
18 2

In [3]:
result.to_csv('./forecast_v01.csv', index=False)
print("Saved — download from the file browser on the left as forecast_v01.csv")

Saved — download from the file browser on the left as forecast_v01.csv


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

PORTFOLIOS = ['A', 'B', 'C', 'D']
DATA_DIR   = './data'
OUTPUT_PATH = './forecast_v02.csv'

DAILY_FEATURES = [
    'day_of_week', 'month', 'day_of_month', 'week_of_month', 'is_weekend',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'cv_lag_364', 'cct_lag_364', 'abd_lag_364'
]

# ══════════════════════════════════════════════════════════════════════════════
# LOAD INTERVAL DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_interval(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Interval'] = df['Interval'].apply(
        lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x)
    )

    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })
    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = (
            df.groupby(['Month', 'Day'])[col]
              .transform(lambda x: x.interpolate(method='linear').bfill().ffill())
              .clip(lower=0)
        )
    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    df['Date']         = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['day_of_month'] = df['Date'].dt.day
    df['month']        = df['Date'].dt.month
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday']   = df['Date'].isin(
        pd.to_datetime(['2025-05-26', '2025-06-19'])
    ).astype(int)
    df['IntervalIdx']  = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )
    return df


# ══════════════════════════════════════════════════════════════════════════════
# LOAD DAILY DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_daily(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Date']         = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df                 = df.sort_values('Date').reset_index(drop=True)
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(
            lambda x: x.fillna(x.median())
        )

    df['dow_sin']    = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']    = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)
    df['cv_lag_364']  = df['Call Volume'].shift(364)
    df['cct_lag_364'] = df['CCT'].shift(364)
    df['abd_lag_364'] = df['Abandon Rate'].shift(364)
    return df


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
aug_days      = pd.date_range('2026-08-01', '2026-08-31', freq='D')
all_forecasts = []

for p in PORTFOLIOS:
    print(f"\n{'='*50}")
    print(f"Portfolio {p}")
    print(f"{'='*50}")

    d  = load_daily(p)
    iv = load_interval(p)

    # Merge daily totals into interval data
    iv = iv.merge(
        d[['Date', 'Call Volume']].rename(columns={'Call Volume': 'daily_cv'}),
        on='Date', how='left'
    )

    # ── STAGE 1: Daily models ─────────────────────────────────────────────────
    train_d = d.dropna(subset=DAILY_FEATURES + ['Call Volume'])
    print(f"Daily training rows: {len(train_d)}")

    daily_models = {}
    for target in ['Call Volume', 'CCT', 'Abandon Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, max_depth=5, random_state=42
        )
        m.fit(train_d[DAILY_FEATURES], train_d[target])
        daily_models[target] = m

    # Build August 2026 daily grid
    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'month':         aug_days.month,
        'day_of_month':  aug_days.day,
        'week_of_month': (aug_days.day - 1) // 7 + 1,
        'is_weekend':    (aug_days.dayofweek >= 5).astype(int),
    })
    aug['dow_sin']   = np.sin(2 * np.pi * aug['day_of_week'] / 7)
    aug['dow_cos']   = np.cos(2 * np.pi * aug['day_of_week'] / 7)
    aug['month_sin'] = np.sin(2 * np.pi * aug['month'] / 12)
    aug['month_cos'] = np.cos(2 * np.pi * aug['month'] / 12)

    aug_2025 = d[
        (d['Date'].dt.month == 8) & (d['Date'].dt.year == 2025)
    ].reset_index(drop=True)
    aug['cv_lag_364']  = aug_2025['Call Volume'].values
    aug['cct_lag_364'] = aug_2025['CCT'].values
    aug['abd_lag_364'] = aug_2025['Abandon Rate'].values

    for target, col in [
        ('Call Volume',  'pred_cv'),
        ('CCT',          'pred_cct'),
        ('Abandon Rate', 'pred_abd')
    ]:
        aug[col] = np.maximum(0, daily_models[target].predict(aug[DAILY_FEATURES]))

    # Slight upward bias on volume — avoids understaffing penalty
    aug['pred_cv'] = aug['pred_cv'] * 1.05

    print("August daily predictions:")
    print(aug[['Date', 'pred_cv', 'pred_cct', 'pred_abd']].to_string())

    # ── STAGE 2: Intraday profiles ────────────────────────────────────────────
    # CV share: what % of daily volume lands in each interval per DOW
    iv['cv_share'] = iv['Call Volume'] / iv['daily_cv'].clip(lower=1)
    profile_cv     = iv.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
    dow_sums       = profile_cv.groupby(level='day_of_week').transform('sum')
    profile_cv     = profile_cv / dow_sums.clip(lower=1e-9)  # normalize to sum=1

    # CCT: raw historical mean per DOW + interval (no model — more stable)
    profile_cct = iv.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean()

    # ABD: historical mean rate per DOW + interval
    profile_abd = iv.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean()

    # Validate on June
    iv_june = iv[iv['month'] == 6].copy()
    iv_june = iv_june.merge(
        profile_cv.rename('cv_share_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    daily_june = iv.groupby('Date')['Call Volume'].sum().rename('actual_daily').reset_index()
    iv_june    = iv_june.merge(daily_june, on='Date', how='left')
    iv_june['cv_pred'] = iv_june['cv_share_pred'] * iv_june['actual_daily']

    mask_cv = iv_june['Call Volume'] > 0
    if mask_cv.sum() > 0:
        mape_cv = mean_absolute_percentage_error(
            iv_june['Call Volume'][mask_cv], iv_june['cv_pred'][mask_cv]
        ) * 100
        print(f"\nCV  profile MAPE on June: {mape_cv:.2f}%")

    iv_june2 = iv[iv['month'] == 6].copy()
    iv_june2 = iv_june2.merge(
        profile_cct.rename('cct_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    mask_cct = iv_june2['CCT'] > 0
    if mask_cct.sum() > 0:
        mape_cct = mean_absolute_percentage_error(
            iv_june2['CCT'][mask_cct], iv_june2['cct_pred'][mask_cct]
        ) * 100
        print(f"CCT profile MAPE on June: {mape_cct:.2f}%")

    iv_june2 = iv_june2.merge(
        profile_abd.rename('abd_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    mask_abd = iv_june2['Abandoned Rate'] > 0
    if mask_abd.sum() > 0:
        mape_abd = mean_absolute_percentage_error(
            iv_june2['Abandoned Rate'][mask_abd], iv_june2['abd_pred'][mask_abd]
        ) * 100
        print(f"ABD profile MAPE on June: {mape_abd:.2f}%")

    # ── Generate August interval forecasts ───────────────────────────────────
    rows = []
    for _, day_row in aug.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])

        for iod in range(48):
            h  = iod // 2
            mn = (iod % 2) * 30

            cv_share = profile_cv.get((dow, iod), 1/48)
            cv_pred  = max(0, cv_share * day_row['pred_cv'])

            cct_pred = max(0, profile_cct.get((dow, iod), day_row['pred_cct']))

            abd_pred = float(np.clip(profile_abd.get((dow, iod), 0.01), 0, 1))

            rows.append({
                'Month':                    'August',
                'Day':                      dom,
                'Interval':                 f"{h}:{mn:02d}",
                f'Calls_Offered_{p}':       round(cv_pred, 2),
                f'Abandoned_Calls_{p}':     round(cv_pred * abd_pred, 2),
                f'Abandoned_Rate_{p}':      round(abd_pred, 6),
                f'CCT_{p}':                 round(cct_pred, 2),
            })

    all_forecasts.append(pd.DataFrame(rows))
    print(f"\n{p}: {len(rows)} interval rows generated")


# ══════════════════════════════════════════════════════════════════════════════
# COMBINE ALL PORTFOLIOS
# ══════════════════════════════════════════════════════════════════════════════
print("\nCombining portfolios...")
result = all_forecasts[0][['Month', 'Day', 'Interval',
    'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A']]

for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(
        all_forecasts[i + 1][[
            'Month', 'Day', 'Interval',
            f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
            f'Abandoned_Rate_{p}', f'CCT_{p}'
        ]],
        on=['Month', 'Day', 'Interval'], how='left'
    )

col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]
result = result[col_order]

# Final clip — no negatives
for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n✅ Saved to {OUTPUT_PATH}")
print(f"   Shape:     {result.shape}  |  Expected: (1488, 19)")
print(f"   Negatives: {(result.select_dtypes('number') < 0).any().any()}")
print(f"   Nulls:     {result.isnull().any().any()}")

print("\n=== Daily CV totals by portfolio ===")
for p in PORTFOLIOS:
    col   = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")

print("\n=== First 5 rows ===")
print(result.head(5).to_string())


Portfolio A
Daily training rows: 367
August daily predictions:
         Date      pred_cv    pred_cct  pred_abd
0  2026-08-01  3861.055106  312.003144  0.013484
1  2026-08-02  2637.884181  323.888684  0.012452
2  2026-08-03  3223.739945  315.657298  0.011858
3  2026-08-04  4657.536004  314.366963  0.014902
4  2026-08-05  4371.311015  319.084548  0.012693
5  2026-08-06  4486.740480  318.600626  0.012705
6  2026-08-07  4613.958442  330.296465  0.009293
7  2026-08-08  3872.597697  333.215748  0.011844
8  2026-08-09  2304.592187  324.519962  0.012326
9  2026-08-10  2972.870038  318.343607  0.011487
10 2026-08-11  4417.937491  320.682554  0.013315
11 2026-08-12  4234.646984  334.413352  0.012433
12 2026-08-13  4216.216743  316.081421  0.017040
13 2026-08-14  4149.951974  316.805116  0.015921
14 2026-08-15  3483.668354  336.145621  0.009167
15 2026-08-16  2175.327058  320.925653  0.010135
16 2026-08-17  2920.307763  319.535181  0.014184
17 2026-08-18  4322.562481  323.854405  0.009441
18 20

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

PORTFOLIOS  = ['A', 'B', 'C', 'D']
DATA_DIR    = './data'
OUTPUT_PATH = './forecast_v03.csv'

DAILY_FEATURES = [
    'day_of_week', 'month', 'day_of_month', 'week_of_month', 'is_weekend',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'cv_lag_364', 'cct_lag_364', 'abd_lag_364'
]

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN INTERVAL DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_interval(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Interval'] = df['Interval'].apply(
        lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x)
    )

    # Build complete skeleton
    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })
    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    # Step 1 — interpolate within each day
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = (
            df.groupby(['Month', 'Day'])[col]
              .transform(lambda x: x.interpolate(method='linear').bfill().ffill())
        )

    # Step 2 — same DOW + interval median
    df['Date']        = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['IntervalIdx'] = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df.groupby(['day_of_week', 'IntervalIdx'])[col].transform(
            lambda x: x.fillna(x.median())
        )

    # Step 3 — overall median fallback
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df[col].fillna(df[col].median())

    # Clip negatives
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df[col].clip(lower=0)
    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    # Time features
    df['day_of_month'] = df['Date'].dt.day
    df['month']        = df['Date'].dt.month
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday']   = df['Date'].isin(
        pd.to_datetime(['2025-05-26', '2025-06-19'])
    ).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['IntervalIdx'] / 48)
    df['hour_cos'] = np.cos(2 * np.pi * df['IntervalIdx'] / 48)
    df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)

    remaining = df[['Call Volume','CCT','Abandoned Calls','Abandoned Rate']].isnull().sum().sum()
    print(f"  {p} interval nulls after cleanup: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN DAILY DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_daily(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Date']         = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df                 = df.sort_values('Date').reset_index(drop=True)
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    # Step 1 — same DOW median
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(
            lambda x: x.fillna(x.median())
        )

    # Step 2 — same month median
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('month')[col].transform(
            lambda x: x.fillna(x.median())
        )

    # Step 3 — overall median fallback
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df[col].fillna(df[col].median())

    df[['Call Volume', 'CCT', 'Abandon Rate']] = df[['Call Volume', 'CCT', 'Abandon Rate']].clip(lower=0)

    df['dow_sin']    = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']    = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)
    df['cv_lag_364']  = df['Call Volume'].shift(364)
    df['cct_lag_364'] = df['CCT'].shift(364)
    df['abd_lag_364'] = df['Abandon Rate'].shift(364)

    remaining = df[['Call Volume', 'CCT', 'Abandon Rate']].isnull().sum().sum()
    print(f"  {p} daily nulls after cleanup: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
aug_days      = pd.date_range('2026-08-01', '2026-08-31', freq='D')
all_forecasts = []

for p in PORTFOLIOS:
    print(f"\n{'='*50}")
    print(f"Portfolio {p}")
    print(f"{'='*50}")

    d  = load_daily(p)
    iv = load_interval(p)
    iv = iv.merge(
        d[['Date', 'Call Volume']].rename(columns={'Call Volume': 'daily_cv'}),
        on='Date', how='left'
    )

    # ── STAGE 1: Daily models ─────────────────────────────────────────────────
    train_d = d.dropna(subset=DAILY_FEATURES + ['Call Volume'])
    daily_models = {}
    for target in ['Call Volume', 'CCT', 'Abandon Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, max_depth=5, random_state=42
        )
        m.fit(train_d[DAILY_FEATURES], train_d[target])
        daily_models[target] = m

    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'month':         aug_days.month,
        'day_of_month':  aug_days.day,
        'week_of_month': (aug_days.day - 1) // 7 + 1,
        'is_weekend':    (aug_days.dayofweek >= 5).astype(int),
    })
    aug['dow_sin']   = np.sin(2 * np.pi * aug['day_of_week'] / 7)
    aug['dow_cos']   = np.cos(2 * np.pi * aug['day_of_week'] / 7)
    aug['month_sin'] = np.sin(2 * np.pi * aug['month'] / 12)
    aug['month_cos'] = np.cos(2 * np.pi * aug['month'] / 12)
    aug_2025 = d[
        (d['Date'].dt.month == 8) & (d['Date'].dt.year == 2025)
    ].reset_index(drop=True)
    aug['cv_lag_364']  = aug_2025['Call Volume'].values
    aug['cct_lag_364'] = aug_2025['CCT'].values
    aug['abd_lag_364'] = aug_2025['Abandon Rate'].values
    for target, col in [
        ('Call Volume',  'pred_cv'),
        ('CCT',          'pred_cct'),
        ('Abandon Rate', 'pred_abd')
    ]:
        aug[col] = np.maximum(0, daily_models[target].predict(aug[DAILY_FEATURES]))

    # 10% upward bias on volume to reduce workload penalty
    aug['pred_cv'] = aug['pred_cv'] * 1.10

    # ── STAGE 2: Intraday profiles — June weighted 2x ─────────────────────────
    iv_weighted = pd.concat([iv, iv[iv['month'] == 6]], ignore_index=True)

    iv_weighted['cv_share'] = iv_weighted['Call Volume'] / iv_weighted['daily_cv'].clip(lower=1)
    profile_cv  = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
    dow_sums    = profile_cv.groupby(level='day_of_week').transform('sum')
    profile_cv  = profile_cv / dow_sums.clip(lower=1e-9)

    profile_cct = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean()
    profile_abd = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean()

    # Validate on June
    iv_june = iv[iv['month'] == 6].copy()
    iv_june = iv_june.merge(
        profile_cv.rename('cv_share_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    daily_june     = iv.groupby('Date')['Call Volume'].sum().rename('actual_daily').reset_index()
    iv_june        = iv_june.merge(daily_june, on='Date', how='left')
    iv_june['cv_pred'] = iv_june['cv_share_pred'] * iv_june['actual_daily']
    mask_cv = iv_june['Call Volume'] > 0
    if mask_cv.sum() > 0:
        mape_cv = mean_absolute_percentage_error(
            iv_june['Call Volume'][mask_cv], iv_june['cv_pred'][mask_cv]
        ) * 100
        print(f"CV  profile MAPE on June: {mape_cv:.2f}%")

    iv_june2 = iv[iv['month'] == 6].copy()
    iv_june2 = iv_june2.merge(
        profile_cct.rename('cct_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    mask_cct = iv_june2['CCT'] > 0
    if mask_cct.sum() > 0:
        mape_cct = mean_absolute_percentage_error(
            iv_june2['CCT'][mask_cct], iv_june2['cct_pred'][mask_cct]
        ) * 100
        print(f"CCT profile MAPE on June: {mape_cct:.2f}%")

    iv_june2 = iv_june2.merge(
        profile_abd.rename('abd_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    mask_abd = iv_june2['Abandoned Rate'] > 0
    if mask_abd.sum() > 0:
        mape_abd = mean_absolute_percentage_error(
            iv_june2['Abandoned Rate'][mask_abd], iv_june2['abd_pred'][mask_abd]
        ) * 100
        print(f"ABD profile MAPE on June: {mape_abd:.2f}%")

    # ── Generate August forecasts ─────────────────────────────────────────────
    rows = []
    for _, day_row in aug.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])

        for iod in range(48):
            h  = iod // 2
            mn = (iod % 2) * 30

            cv_share = profile_cv.get((dow, iod), 1/48)
            cv_pred  = max(0, cv_share * day_row['pred_cv'])
            cct_pred = max(0, profile_cct.get((dow, iod), day_row['pred_cct']))
            abd_pred = float(np.clip(profile_abd.get((dow, iod), 0.01), 0, 1))

            rows.append({
                'Month':                 'August',
                'Day':                   dom,
                'Interval':              f"{h}:{mn:02d}",
                f'Calls_Offered_{p}':    round(cv_pred, 2),
                f'Abandoned_Calls_{p}':  round(cv_pred * abd_pred, 2),
                f'Abandoned_Rate_{p}':   round(abd_pred, 6),
                f'CCT_{p}':              round(cct_pred, 2),
            })

    all_forecasts.append(pd.DataFrame(rows))
    print(f"{p}: {len(rows)} rows generated")


# ══════════════════════════════════════════════════════════════════════════════
# COMBINE
# ══════════════════════════════════════════════════════════════════════════════
print("\nCombining portfolios...")
result = all_forecasts[0][['Month', 'Day', 'Interval',
    'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A']]
for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(
        all_forecasts[i + 1][[
            'Month', 'Day', 'Interval',
            f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
            f'Abandoned_Rate_{p}', f'CCT_{p}'
        ]],
        on=['Month', 'Day', 'Interval'], how='left'
    )

col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]
result = result[col_order]

for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n✅ Saved to {OUTPUT_PATH}")
print(f"   Shape:     {result.shape}  |  Expected: (1488, 19)")
print(f"   Negatives: {(result.select_dtypes('number') < 0).any().any()}")
print(f"   Nulls:     {result.isnull().any().any()}")
print("\n=== Daily CV totals by portfolio ===")
for p in PORTFOLIOS:
    col   = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")
print("\n=== First 5 rows ===")
print(result.head(5).to_string())


Portfolio A
  A daily nulls after cleanup: 0
  A interval nulls after cleanup: 0
CV  profile MAPE on June: 26.68%
CCT profile MAPE on June: 26.70%
ABD profile MAPE on June: 51.48%
A: 1488 rows generated

Portfolio B
  B daily nulls after cleanup: 0
  B interval nulls after cleanup: 0
CV  profile MAPE on June: 20.16%
CCT profile MAPE on June: 18.37%
ABD profile MAPE on June: 89.57%
B: 1488 rows generated

Portfolio C
  C daily nulls after cleanup: 0
  C interval nulls after cleanup: 0
CV  profile MAPE on June: 9.99%
CCT profile MAPE on June: 8.09%
ABD profile MAPE on June: 143.97%
C: 1488 rows generated

Portfolio D
  D daily nulls after cleanup: 0
  D interval nulls after cleanup: 0
CV  profile MAPE on June: 13.42%
CCT profile MAPE on June: 15.58%
ABD profile MAPE on June: 111.55%
D: 1488 rows generated

Combining portfolios...

✅ Saved to ./forecast_v03.csv
   Shape:     (1488, 19)  |  Expected: (1488, 19)
   Negatives: False
   Nulls:     False

=== Daily CV totals by portfolio ===


In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

PORTFOLIOS  = ['A', 'B', 'C', 'D']
DATA_DIR    = './data'
OUTPUT_PATH = './forecast_v04.csv'

DAILY_FEATURES = [
    'day_of_week', 'month', 'day_of_month', 'week_of_month', 'is_weekend',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'cv_lag_364', 'cct_lag_364', 'abd_lag_364',
    'aug_cv_mean', 'aug_cct_mean', 'aug_abd_mean',  # NEW — August baseline
    'aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean'  # NEW — August DOW mean
]

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN INTERVAL DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_interval(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Interval'] = df['Interval'].apply(
        lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x)
    )
    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })
    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    # Step 1 — interpolate within each day
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = (
            df.groupby(['Month', 'Day'])[col]
              .transform(lambda x: x.interpolate(method='linear').bfill().ffill())
        )

    # Step 2 — same DOW + interval median
    df['Date']        = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['IntervalIdx'] = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df.groupby(['day_of_week', 'IntervalIdx'])[col].transform(
            lambda x: x.fillna(x.median())
        )

    # Step 3 — overall median fallback
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)
    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    # Time features
    df['day_of_month'] = df['Date'].dt.day
    df['month']        = df['Date'].dt.month
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday']   = df['Date'].isin(
        pd.to_datetime(['2025-05-26', '2025-06-19'])
    ).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['IntervalIdx'] / 48)
    df['hour_cos'] = np.cos(2 * np.pi * df['IntervalIdx'] / 48)
    df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)

    remaining = df[['Call Volume','CCT','Abandoned Calls','Abandoned Rate']].isnull().sum().sum()
    print(f"  {p} interval nulls after cleanup: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN DAILY DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_daily(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Date']         = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df                 = df.sort_values('Date').reset_index(drop=True)
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    # Three-level null fill
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('month')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df[col].fillna(df[col].median())
    df[['Call Volume', 'CCT', 'Abandon Rate']] = df[['Call Volume', 'CCT', 'Abandon Rate']].clip(lower=0)

    # Cyclical features
    df['dow_sin']   = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Lag features
    df['cv_lag_364']  = df['Call Volume'].shift(364)
    df['cct_lag_364'] = df['CCT'].shift(364)
    df['abd_lag_364'] = df['Abandon Rate'].shift(364)

    # NEW — August overall mean (both years)
    aug_data = df[df['month'] == 8]
    df['aug_cv_mean']  = aug_data['Call Volume'].mean()
    df['aug_cct_mean'] = aug_data['CCT'].mean()
    df['aug_abd_mean'] = aug_data['Abandon Rate'].mean()

    # NEW — August mean by day of week
    aug_dow = aug_data.groupby('day_of_week')[['Call Volume','CCT','Abandon Rate']].mean()
    aug_dow.columns = ['aug_dow_cv_mean','aug_dow_cct_mean','aug_dow_abd_mean']
    df = df.merge(aug_dow, on='day_of_week', how='left')

    # Fill aug features for non-August rows too (model needs them for all rows)
    for col in ['aug_cv_mean','aug_cct_mean','aug_abd_mean',
                'aug_dow_cv_mean','aug_dow_cct_mean','aug_dow_abd_mean']:
        df[col] = df[col].fillna(df[col].median())

    remaining = df[['Call Volume', 'CCT', 'Abandon Rate']].isnull().sum().sum()
    print(f"  {p} daily nulls after cleanup: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
aug_days      = pd.date_range('2026-08-01', '2026-08-31', freq='D')
all_forecasts = []

for p in PORTFOLIOS:
    print(f"\n{'='*50}")
    print(f"Portfolio {p}")
    print(f"{'='*50}")

    d  = load_daily(p)
    iv = load_interval(p)
    iv = iv.merge(
        d[['Date', 'Call Volume']].rename(columns={'Call Volume': 'daily_cv'}),
        on='Date', how='left'
    )

    # ── STAGE 1: Daily models ─────────────────────────────────────────────────
    train_d = d.dropna(subset=DAILY_FEATURES + ['Call Volume'])
    print(f"Daily training rows: {len(train_d)}")

    daily_models = {}
    for target in ['Call Volume', 'CCT', 'Abandon Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05, max_depth=5, random_state=42
        )
        m.fit(train_d[DAILY_FEATURES], train_d[target])
        daily_models[target] = m

    # Build August 2026 daily grid
    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'month':         aug_days.month,
        'day_of_month':  aug_days.day,
        'week_of_month': (aug_days.day - 1) // 7 + 1,
        'is_weekend':    (aug_days.dayofweek >= 5).astype(int),
    })
    aug['dow_sin']   = np.sin(2 * np.pi * aug['day_of_week'] / 7)
    aug['dow_cos']   = np.cos(2 * np.pi * aug['day_of_week'] / 7)
    aug['month_sin'] = np.sin(2 * np.pi * aug['month'] / 12)
    aug['month_cos'] = np.cos(2 * np.pi * aug['month'] / 12)

    aug_2025 = d[
        (d['Date'].dt.month == 8) & (d['Date'].dt.year == 2025)
    ].reset_index(drop=True)
    aug['cv_lag_364']  = aug_2025['Call Volume'].values
    aug['cct_lag_364'] = aug_2025['CCT'].values
    aug['abd_lag_364'] = aug_2025['Abandon Rate'].values

    # August baseline features
    aug_all = d[d['month'] == 8]
    aug['aug_cv_mean']  = aug_all['Call Volume'].mean()
    aug['aug_cct_mean'] = aug_all['CCT'].mean()
    aug['aug_abd_mean'] = aug_all['Abandon Rate'].mean()

    aug_dow_means = aug_all.groupby('day_of_week')[['Call Volume','CCT','Abandon Rate']].mean()
    aug_dow_means.columns = ['aug_dow_cv_mean','aug_dow_cct_mean','aug_dow_abd_mean']
    aug = aug.merge(aug_dow_means, on='day_of_week', how='left')

    for target, col in [
        ('Call Volume',  'pred_cv'),
        ('CCT',          'pred_cct'),
        ('Abandon Rate', 'pred_abd')
    ]:
        aug[col] = np.maximum(0, daily_models[target].predict(aug[DAILY_FEATURES]))

    # 15% upward bias — avoids understaffing workload penalty
    aug['pred_cv'] = aug['pred_cv'] * 1.15

    print("August daily predictions:")
    print(aug[['Date', 'pred_cv', 'pred_cct', 'pred_abd']].to_string())

    # ── STAGE 2: Intraday profiles — June weighted 2x ─────────────────────────
    iv_weighted = pd.concat([iv, iv[iv['month'] == 6]], ignore_index=True)

    iv_weighted['cv_share'] = iv_weighted['Call Volume'] / iv_weighted['daily_cv'].clip(lower=1)
    profile_cv  = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
    dow_sums    = profile_cv.groupby(level='day_of_week').transform('sum')
    profile_cv  = profile_cv / dow_sums.clip(lower=1e-9)

    profile_cct = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean()
    profile_abd = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean()

    # Validate on June
    iv_june = iv[iv['month'] == 6].copy()
    iv_june = iv_june.merge(
        profile_cv.rename('cv_share_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    daily_june     = iv.groupby('Date')['Call Volume'].sum().rename('actual_daily').reset_index()
    iv_june        = iv_june.merge(daily_june, on='Date', how='left')
    iv_june['cv_pred'] = iv_june['cv_share_pred'] * iv_june['actual_daily']
    mask_cv = iv_june['Call Volume'] > 0
    if mask_cv.sum() > 0:
        mape_cv = mean_absolute_percentage_error(
            iv_june['Call Volume'][mask_cv], iv_june['cv_pred'][mask_cv]
        ) * 100
        print(f"CV  profile MAPE on June: {mape_cv:.2f}%")

    iv_june2 = iv[iv['month'] == 6].copy()
    iv_june2 = iv_june2.merge(
        profile_cct.rename('cct_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    mask_cct = iv_june2['CCT'] > 0
    if mask_cct.sum() > 0:
        mape_cct = mean_absolute_percentage_error(
            iv_june2['CCT'][mask_cct], iv_june2['cct_pred'][mask_cct]
        ) * 100
        print(f"CCT profile MAPE on June: {mape_cct:.2f}%")

    iv_june2 = iv_june2.merge(
        profile_abd.rename('abd_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    mask_abd = iv_june2['Abandoned Rate'] > 0
    if mask_abd.sum() > 0:
        mape_abd = mean_absolute_percentage_error(
            iv_june2['Abandoned Rate'][mask_abd], iv_june2['abd_pred'][mask_abd]
        ) * 100
        print(f"ABD profile MAPE on June: {mape_abd:.2f}%")

    # ── Generate August forecasts ─────────────────────────────────────────────
    rows = []
    for _, day_row in aug.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])

        for iod in range(48):
            h  = iod // 2
            mn = (iod % 2) * 30

            cv_share = profile_cv.get((dow, iod), 1/48)
            cv_pred  = max(0, cv_share * day_row['pred_cv'])
            cct_pred = max(0, profile_cct.get((dow, iod), day_row['pred_cct']))
            abd_pred = float(np.clip(profile_abd.get((dow, iod), 0.01), 0, 1))

            rows.append({
                'Month':                 'August',
                'Day':                   dom,
                'Interval':              f"{h}:{mn:02d}",
                f'Calls_Offered_{p}':    round(cv_pred, 2),
                f'Abandoned_Calls_{p}':  round(cv_pred * abd_pred, 2),
                f'Abandoned_Rate_{p}':   round(abd_pred, 6),
                f'CCT_{p}':              round(cct_pred, 2),
            })

    all_forecasts.append(pd.DataFrame(rows))
    print(f"{p}: {len(rows)} rows generated")


# ══════════════════════════════════════════════════════════════════════════════
# COMBINE
# ══════════════════════════════════════════════════════════════════════════════
print("\nCombining portfolios...")
result = all_forecasts[0][['Month', 'Day', 'Interval',
    'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A']]
for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(
        all_forecasts[i + 1][[
            'Month', 'Day', 'Interval',
            f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
            f'Abandoned_Rate_{p}', f'CCT_{p}'
        ]],
        on=['Month', 'Day', 'Interval'], how='left'
    )

col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]
result = result[col_order]

for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n✅ Saved to {OUTPUT_PATH}")
print(f"   Shape:     {result.shape}  |  Expected: (1488, 19)")
print(f"   Negatives: {(result.select_dtypes('number') < 0).any().any()}")
print(f"   Nulls:     {result.isnull().any().any()}")
print("\n=== Daily CV totals by portfolio ===")
for p in PORTFOLIOS:
    col   = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")
print("\n=== First 5 rows ===")
print(result.head(5).to_string())


Portfolio A
  A daily nulls after cleanup: 0
  A interval nulls after cleanup: 0
Daily training rows: 367
August daily predictions:
         Date      pred_cv    pred_cct  pred_abd
0  2026-08-01  4142.063938  311.065424  0.012616
1  2026-08-02  2880.802337  321.283546  0.010952
2  2026-08-03  3403.126587  317.277671  0.011820
3  2026-08-04  5078.800070  309.395750  0.014596
4  2026-08-05  4696.308254  318.830120  0.010400
5  2026-08-06  4946.857594  321.695702  0.012990
6  2026-08-07  4939.895902  326.870709  0.007357
7  2026-08-08  4281.595414  330.296679  0.010993
8  2026-08-09  2559.387280  323.517373  0.011227
9  2026-08-10  3250.077675  317.324377  0.012176
10 2026-08-11  4854.962660  321.613161  0.013431
11 2026-08-12  4593.347242  333.893074  0.010704
12 2026-08-13  4705.206218  317.988430  0.016509
13 2026-08-14  4537.728081  315.800766  0.014197
14 2026-08-15  3772.744635  335.442009  0.010185
15 2026-08-16  2385.324502  321.186958  0.010305
16 2026-08-17  3181.905403  318.37

In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

PORTFOLIOS  = ['A', 'B', 'C', 'D']
DATA_DIR    = './data'
OUTPUT_PATH = './forecast_v05.csv'

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN INTERVAL DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_interval(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Interval'] = df['Interval'].apply(
        lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x)
    )
    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })
    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = (
            df.groupby(['Month', 'Day'])[col]
              .transform(lambda x: x.interpolate(method='linear').bfill().ffill())
        )
    df['Date']        = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['IntervalIdx'] = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df.groupby(['day_of_week', 'IntervalIdx'])[col].transform(
            lambda x: x.fillna(x.median())
        )
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)
    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    df['day_of_month'] = df['Date'].dt.day
    df['month']        = df['Date'].dt.month
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    remaining = df[['Call Volume','CCT','Abandoned Calls','Abandoned Rate']].isnull().sum().sum()
    print(f"  {p} interval nulls: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN DAILY DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_daily(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Date']         = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df                 = df.sort_values('Date').reset_index(drop=True)
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('month')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)

    remaining = df[['Call Volume', 'CCT', 'Abandon Rate']].isnull().sum().sum()
    print(f"  {p} daily nulls: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
aug_days      = pd.date_range('2026-08-01', '2026-08-31', freq='D')
all_forecasts = []

for p in PORTFOLIOS:
    print(f"\n{'='*50}")
    print(f"Portfolio {p}")
    print(f"{'='*50}")

    d  = load_daily(p)
    iv = load_interval(p)
    iv = iv.merge(
        d[['Date', 'Call Volume']].rename(columns={'Call Volume': 'daily_cv'}),
        on='Date', how='left'
    )

    # ── STAGE 1: Use August actuals directly ──────────────────────────────────
    # Average Aug 2024 + Aug 2025 by day of week — best predictor of Aug 2026
    aug_actual = d[d['month'] == 8].copy()

    aug_dow_cv  = aug_actual.groupby('day_of_week')['Call Volume'].mean()
    aug_dow_cct = aug_actual.groupby('day_of_week')['CCT'].mean()
    aug_dow_abd = aug_actual.groupby('day_of_week')['Abandon Rate'].mean()

    # Also compute day-of-month trend within August
    # (some days of month consistently higher — billing cycle)
    aug_dom_factor = aug_actual.groupby('day_of_month')['Call Volume'].mean()
    aug_dom_factor = aug_dom_factor / aug_dom_factor.mean()  # normalize to 1.0

    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'day_of_month':  aug_days.day,
    })

    # Base prediction = Aug DOW mean × day-of-month scaling factor
    aug['pred_cv'] = aug['day_of_week'].map(aug_dow_cv)
    aug['pred_cct']= aug['day_of_week'].map(aug_dow_cct)
    aug['pred_abd']= aug['day_of_week'].map(aug_dow_abd)

    # Apply day-of-month scaling
    aug['dom_factor'] = aug['day_of_month'].map(aug_dom_factor).fillna(1.0)
    aug['pred_cv']    = aug['pred_cv'] * aug['dom_factor']

    # 10% upward bias to avoid understaffing penalty
    aug['pred_cv'] = aug['pred_cv'] * 1.10

    # Clip
    aug['pred_cv']  = aug['pred_cv'].clip(lower=0)
    aug['pred_cct'] = aug['pred_cct'].clip(lower=0)
    aug['pred_abd'] = aug['pred_abd'].clip(lower=0)

    print("August daily predictions (direct from actuals):")
    print(aug[['Date','pred_cv','pred_cct','pred_abd']].to_string())

    # ── STAGE 2: Intraday profiles — June weighted 2x ─────────────────────────
    iv_weighted = pd.concat([iv, iv[iv['month'] == 6]], ignore_index=True)

    iv_weighted['cv_share'] = iv_weighted['Call Volume'] / iv_weighted['daily_cv'].clip(lower=1)
    profile_cv  = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
    dow_sums    = profile_cv.groupby(level='day_of_week').transform('sum')
    profile_cv  = profile_cv / dow_sums.clip(lower=1e-9)

    profile_cct = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean()
    profile_abd = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean()

    # Validate profiles on June
    iv_june = iv[iv['month'] == 6].copy()
    iv_june = iv_june.merge(
        profile_cv.rename('cv_share_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    daily_june     = iv.groupby('Date')['Call Volume'].sum().rename('actual_daily').reset_index()
    iv_june        = iv_june.merge(daily_june, on='Date', how='left')
    iv_june['cv_pred'] = iv_june['cv_share_pred'] * iv_june['actual_daily']
    mask_cv = iv_june['Call Volume'] > 0
    if mask_cv.sum() > 0:
        mape_cv = mean_absolute_percentage_error(
            iv_june['Call Volume'][mask_cv], iv_june['cv_pred'][mask_cv]
        ) * 100
        print(f"CV  profile MAPE on June: {mape_cv:.2f}%")

    iv_june2 = iv[iv['month'] == 6].copy()
    iv_june2 = iv_june2.merge(
        profile_cct.rename('cct_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    mask_cct = iv_june2['CCT'] > 0
    if mask_cct.sum() > 0:
        mape_cct = mean_absolute_percentage_error(
            iv_june2['CCT'][mask_cct], iv_june2['cct_pred'][mask_cct]
        ) * 100
        print(f"CCT profile MAPE on June: {mape_cct:.2f}%")

    iv_june2 = iv_june2.merge(
        profile_abd.rename('abd_pred').reset_index(),
        on=['day_of_week', 'IntervalIdx'], how='left'
    )
    mask_abd = iv_june2['Abandoned Rate'] > 0
    if mask_abd.sum() > 0:
        mape_abd = mean_absolute_percentage_error(
            iv_june2['Abandoned Rate'][mask_abd], iv_june2['abd_pred'][mask_abd]
        ) * 100
        print(f"ABD profile MAPE on June: {mape_abd:.2f}%")

    # ── Generate August forecasts ─────────────────────────────────────────────
    rows = []
    for _, day_row in aug.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])

        for iod in range(48):
            h  = iod // 2
            mn = (iod % 2) * 30

            cv_share = profile_cv.get((dow, iod), 1/48)
            cv_pred  = max(0, cv_share * day_row['pred_cv'])
            cct_pred = max(0, profile_cct.get((dow, iod), day_row['pred_cct']))
            abd_pred = float(np.clip(profile_abd.get((dow, iod), 0.01), 0, 1))

            rows.append({
                'Month':                 'August',
                'Day':                   dom,
                'Interval':              f"{h}:{mn:02d}",
                f'Calls_Offered_{p}':    round(cv_pred, 2),
                f'Abandoned_Calls_{p}':  round(cv_pred * abd_pred, 2),
                f'Abandoned_Rate_{p}':   round(abd_pred, 6),
                f'CCT_{p}':              round(cct_pred, 2),
            })

    all_forecasts.append(pd.DataFrame(rows))
    print(f"{p}: {len(rows)} rows generated")


# ══════════════════════════════════════════════════════════════════════════════
# COMBINE
# ══════════════════════════════════════════════════════════════════════════════
print("\nCombining portfolios...")
result = all_forecasts[0][['Month', 'Day', 'Interval',
    'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A']]
for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(
        all_forecasts[i + 1][[
            'Month', 'Day', 'Interval',
            f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
            f'Abandoned_Rate_{p}', f'CCT_{p}'
        ]],
        on=['Month', 'Day', 'Interval'], how='left'
    )

col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]
result = result[col_order]

for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n✅ Saved to {OUTPUT_PATH}")
print(f"   Shape:     {result.shape}  |  Expected: (1488, 19)")
print(f"   Negatives: {(result.select_dtypes('number') < 0).any().any()}")
print(f"   Nulls:     {result.isnull().any().any()}")
print("\n=== Daily CV totals by portfolio ===")
for p in PORTFOLIOS:
    col   = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")
print("\n=== First 5 rows ===")
print(result.head(5).to_string())


Portfolio A
  A daily nulls: 0
  A interval nulls: 0
August daily predictions (direct from actuals):
         Date      pred_cv    pred_cct  pred_abd
0  2026-08-01  4534.904685  314.942000  0.013720
1  2026-08-02  1748.602772  313.992222  0.011944
2  2026-08-03  3433.577541  317.516250  0.013437
3  2026-08-04  4292.388463  310.961250  0.015838
4  2026-08-05  6461.401049  326.853750  0.009600
5  2026-08-06  5723.145039  324.661111  0.013867
6  2026-08-07  5757.548933  323.739000  0.012440
7  2026-08-08  3979.793845  314.942000  0.013720
8  2026-08-09  1474.225748  313.992222  0.011944
9  2026-08-10  3186.506789  317.516250  0.013437
10 2026-08-11  4110.579387  310.961250  0.015838
11 2026-08-12  5493.146812  326.853750  0.009600
12 2026-08-13  5344.667970  324.661111  0.013867
13 2026-08-14  5593.413908  323.739000  0.012440
14 2026-08-15  3800.305008  314.942000  0.013720
15 2026-08-16  1546.324820  313.992222  0.011944
16 2026-08-17  3484.403524  317.516250  0.013437
17 2026-08-18  3

In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

PORTFOLIOS  = ['A', 'B', 'C', 'D']
DATA_DIR    = './data'
OUTPUT_PATH = './forecast_v06.csv'

INTERVAL_FEATURES = [
    'IntervalIdx', 'day_of_week', 'day_of_month', 'month', 'week_of_month',
    'is_weekend', 'is_holiday', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'daily_cv'  # August actual anchor
]

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN INTERVAL DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_interval(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Interval'] = df['Interval'].apply(
        lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x)
    )
    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })
    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    # Step 1 — interpolate within each day
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = (
            df.groupby(['Month', 'Day'])[col]
              .transform(lambda x: x.interpolate(method='linear').bfill().ffill())
        )

    # Step 2 — same DOW + interval median
    df['Date']        = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['IntervalIdx'] = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df.groupby(['day_of_week', 'IntervalIdx'])[col].transform(
            lambda x: x.fillna(x.median())
        )

    # Step 3 — overall median fallback
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)
    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    # Time features
    df['day_of_month'] = df['Date'].dt.day
    df['month']        = df['Date'].dt.month
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday']   = df['Date'].isin(
        pd.to_datetime(['2025-05-26', '2025-06-19'])
    ).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['IntervalIdx'] / 48)
    df['hour_cos'] = np.cos(2 * np.pi * df['IntervalIdx'] / 48)
    df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)

    remaining = df[['Call Volume','CCT','Abandoned Calls','Abandoned Rate']].isnull().sum().sum()
    print(f"  {p} interval nulls: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN DAILY DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_daily(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Date']         = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df                 = df.sort_values('Date').reset_index(drop=True)
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('month')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)

    remaining = df[['Call Volume', 'CCT', 'Abandon Rate']].isnull().sum().sum()
    print(f"  {p} daily nulls: {remaining}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
aug_days      = pd.date_range('2026-08-01', '2026-08-31', freq='D')
all_forecasts = []

for p in PORTFOLIOS:
    print(f"\n{'='*50}")
    print(f"Portfolio {p}")
    print(f"{'='*50}")

    d  = load_daily(p)
    iv = load_interval(p)

    # ── Merge actual daily totals into interval training data ─────────────────
    iv = iv.merge(
        d[['Date', 'Call Volume']].rename(columns={'Call Volume': 'daily_cv'}),
        on='Date', how='left'
    )

    # ── Build August daily anchor from Aug 2024 + Aug 2025 actuals ────────────
    aug_actual  = d[d['month'] == 8].copy()
    aug_dow_cv  = aug_actual.groupby('day_of_week')['Call Volume'].mean()
    aug_dow_cct = aug_actual.groupby('day_of_week')['CCT'].mean()
    aug_dow_abd = aug_actual.groupby('day_of_week')['Abandon Rate'].mean()

    # Day-of-month scaling factor within August
    aug_dom_factor = aug_actual.groupby('day_of_month')['Call Volume'].mean()
    aug_dom_factor = aug_dom_factor / aug_dom_factor.mean()

    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'day_of_month':  aug_days.day,
        'week_of_month': (aug_days.day - 1) // 7 + 1,
        'is_weekend':    (aug_days.dayofweek >= 5).astype(int),
        'month':         8,
        'is_holiday':    0,
    })
    aug['pred_cv']  = aug['day_of_week'].map(aug_dow_cv)
    aug['pred_cct'] = aug['day_of_week'].map(aug_dow_cct)
    aug['pred_abd'] = aug['day_of_week'].map(aug_dow_abd)
    aug['dom_factor']= aug['day_of_month'].map(aug_dom_factor).fillna(1.0)
    aug['pred_cv']   = (aug['pred_cv'] * aug['dom_factor']).clip(lower=0)

    # 10% upward bias — avoids understaffing workload penalty
    aug['pred_cv'] = aug['pred_cv'] * 1.10

    print("August daily anchor (from Aug 2024+2025 actuals):")
    print(aug[['Date','pred_cv','pred_cct','pred_abd']].to_string())

    # ── Train interval models with daily_cv as anchor feature ─────────────────
    # Train on Apr+May, validate on June
    train_iv = iv[iv['month'].isin([4, 5])].dropna(
        subset=INTERVAL_FEATURES + ['Call Volume', 'CCT', 'Abandoned Rate']
    )
    val_iv = iv[iv['month'] == 6].dropna(subset=INTERVAL_FEATURES + ['Call Volume'])
    print(f"\nInterval training rows: {len(train_iv)} | Val rows: {len(val_iv)}")

    iv_models = {}
    for target in ['Call Volume', 'CCT', 'Abandoned Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=300, learning_rate=0.05,
            max_depth=6, min_samples_leaf=20, random_state=42
        )
        m.fit(train_iv[INTERVAL_FEATURES], train_iv[target])

        preds = np.maximum(0, m.predict(val_iv[INTERVAL_FEATURES]))
        mask  = val_iv[target] > 0
        if mask.sum() > 0:
            mape = mean_absolute_percentage_error(
                val_iv[target][mask], preds[mask]
            ) * 100
            print(f"  {target}: Val MAPE = {mape:.2f}%")

        iv_models[target] = m

    # ── Also build intraday profile as fallback ───────────────────────────────
    iv_weighted = pd.concat([iv, iv[iv['month'] == 6]], ignore_index=True)
    iv_weighted['cv_share'] = iv_weighted['Call Volume'] / iv_weighted['daily_cv'].clip(lower=1)
    profile_cv  = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
    dow_sums    = profile_cv.groupby(level='day_of_week').transform('sum')
    profile_cv  = profile_cv / dow_sums.clip(lower=1e-9)
    profile_cct = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean()
    profile_abd = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean()

    # ── Generate August forecasts ─────────────────────────────────────────────
    rows = []
    for _, day_row in aug.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])
        wom = int((dom - 1) // 7 + 1)

        # Build full interval grid for this day — vectorised
        iods = np.arange(48)
        feat = pd.DataFrame({
            'IntervalIdx':   iods,
            'day_of_week':   dow,
            'day_of_month':  dom,
            'month':         8,
            'week_of_month': wom,
            'is_weekend':    int(dow >= 5),
            'is_holiday':    0,
            'hour_sin':      np.sin(2 * np.pi * iods / 48),
            'hour_cos':      np.cos(2 * np.pi * iods / 48),
            'dow_sin':       np.sin(2 * np.pi * dow / 7),
            'dow_cos':       np.cos(2 * np.pi * dow / 7),
            'daily_cv':      day_row['pred_cv'],  # August actual anchor
        })

        cv_preds  = np.maximum(0, iv_models['Call Volume'].predict(feat[INTERVAL_FEATURES]))
        cct_preds = np.maximum(0, iv_models['CCT'].predict(feat[INTERVAL_FEATURES]))
        abd_preds = np.clip(iv_models['Abandoned Rate'].predict(feat[INTERVAL_FEATURES]), 0, 1)

        # Scale CV so daily total matches anchor exactly
        cv_sum = cv_preds.sum()
        if cv_sum > 0:
            cv_preds = cv_preds * (day_row['pred_cv'] / cv_sum)

        for iod in range(48):
            h  = iod // 2
            mn = (iod % 2) * 30
            cv  = max(0, cv_preds[iod])
            cct = max(0, cct_preds[iod])
            abd = float(np.clip(abd_preds[iod], 0, 1))

            rows.append({
                'Month':                 'August',
                'Day':                   dom,
                'Interval':              f"{h}:{mn:02d}",
                f'Calls_Offered_{p}':    round(cv, 2),
                f'Abandoned_Calls_{p}':  round(cv * abd, 2),
                f'Abandoned_Rate_{p}':   round(abd, 6),
                f'CCT_{p}':              round(cct, 2),
            })

    all_forecasts.append(pd.DataFrame(rows))
    print(f"{p}: {len(rows)} rows generated")


# ══════════════════════════════════════════════════════════════════════════════
# COMBINE
# ══════════════════════════════════════════════════════════════════════════════
print("\nCombining portfolios...")
result = all_forecasts[0][['Month', 'Day', 'Interval',
    'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A']]
for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(
        all_forecasts[i + 1][[
            'Month', 'Day', 'Interval',
            f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
            f'Abandoned_Rate_{p}', f'CCT_{p}'
        ]],
        on=['Month', 'Day', 'Interval'], how='left'
    )

col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]
result = result[col_order]

for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n✅ Saved to {OUTPUT_PATH}")
print(f"   Shape:     {result.shape}  |  Expected: (1488, 19)")
print(f"   Negatives: {(result.select_dtypes('number') < 0).any().any()}")
print(f"   Nulls:     {result.isnull().any().any()}")
print("\n=== Daily CV totals by portfolio ===")
for p in PORTFOLIOS:
    col   = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")
print("\n=== First 5 rows ===")
print(result.head(5).to_string())


Portfolio A
  A daily nulls: 0
  A interval nulls: 0
August daily anchor (from Aug 2024+2025 actuals):
         Date      pred_cv    pred_cct  pred_abd
0  2026-08-01  4534.904685  314.942000  0.013720
1  2026-08-02  1748.602772  313.992222  0.011944
2  2026-08-03  3433.577541  317.516250  0.013437
3  2026-08-04  4292.388463  310.961250  0.015838
4  2026-08-05  6461.401049  326.853750  0.009600
5  2026-08-06  5723.145039  324.661111  0.013867
6  2026-08-07  5757.548933  323.739000  0.012440
7  2026-08-08  3979.793845  314.942000  0.013720
8  2026-08-09  1474.225748  313.992222  0.011944
9  2026-08-10  3186.506789  317.516250  0.013437
10 2026-08-11  4110.579387  310.961250  0.015838
11 2026-08-12  5493.146812  326.853750  0.009600
12 2026-08-13  5344.667970  324.661111  0.013867
13 2026-08-14  5593.413908  323.739000  0.012440
14 2026-08-15  3800.305008  314.942000  0.013720
15 2026-08-16  1546.324820  313.992222  0.011944
16 2026-08-17  3484.403524  317.516250  0.013437
17 2026-08-18 

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings("ignore")

PORTFOLIOS = ["A", "B", "C", "D"]
DATA_DIR = "./data"
OUTPUT_PATH = "./forecast_v07.csv"

DAILY_FEATURES = [
    "day_of_week", "day_of_month", "month", "week_of_month",
    "is_weekend", "trend", "cv_lag_1", "cv_lag_7",
    "cv_roll_7", "cv_roll_14"
]

INTERVAL_FEATURES = [
    "IntervalIdx", "day_of_week", "day_of_month", "month", "week_of_month",
    "is_weekend", "is_holiday", "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "daily_cv",
    "cv_lag_1", "cv_lag_2", "cv_lag_48", "cv_lag_336",
    "cv_roll_3", "cv_roll_6", "cv_roll_12",
    "daily_cv_lag_1",
    "hist_interval_share"
]

def safe_mape(y_true, y_pred):
    mask = y_true > 0
    if mask.sum() == 0:
        return np.nan
    return mean_absolute_percentage_error(y_true[mask], y_pred[mask]) * 100

def load_daily(portfolio):
    df = pd.read_csv(f"{DATA_DIR}/{portfolio} - Daily.csv", index_col=0)
    df.columns = df.columns.str.strip()

    df["Date"] = pd.to_datetime(df["Date"].astype(str).str[:8], format="%m/%d/%y")
    df = df.sort_values("Date").reset_index(drop=True)

    df["day_of_week"] = df["Date"].dt.dayofweek
    df["month"] = df["Date"].dt.month
    df["day_of_month"] = df["Date"].dt.day
    df["week_of_month"] = (df["day_of_month"] - 1) // 7 + 1
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    for col in ["Call Volume", "CCT", "Abandon Rate"]:
        df[col] = df.groupby("day_of_week")[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df.groupby("month")[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)

    return df

def load_interval(portfolio):
    df = pd.read_csv(f"{DATA_DIR}/{portfolio} - Interval.csv", index_col=0)
    df.columns = df.columns.str.strip()

    df["Interval"] = df["Interval"].apply(
        lambda x: x.strftime("%H:%M:%S") if hasattr(x, "strftime") else str(x)
    )

    dr = pd.date_range("2025-04-01", "2025-06-30 23:30", freq="30min")
    skeleton = pd.DataFrame({
        "Month": dr.month_name(),
        "Day": dr.day,
        "Interval": dr.strftime("%H:%M:%S")
    })

    df = skeleton.merge(df, on=["Month", "Day", "Interval"], how="left")

    for col in ["Call Volume", "CCT", "Abandoned Calls", "Abandoned Rate"]:
        df[col] = (
            df.groupby(["Month", "Day"])[col]
              .transform(lambda x: x.interpolate(method="linear").bfill().ffill())
        )

    df["Date"] = pd.to_datetime(df["Month"] + " " + df["Day"].astype(str) + " 2025")
    df["day_of_week"] = df["Date"].dt.dayofweek
    df["IntervalIdx"] = df["Interval"].apply(lambda x: int(x.split(":")[0]) * 2 + int(x.split(":")[1]) // 30)

    for col in ["Call Volume", "CCT", "Abandoned Calls", "Abandoned Rate"]:
        df[col] = df.groupby(["day_of_week", "IntervalIdx"])[col].transform(lambda x: x.fillna(x.median()))
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)

    df["Abandoned Rate"] = df["Abandoned Rate"].clip(0, 1)

    df["day_of_month"] = df["Date"].dt.day
    df["month"] = df["Date"].dt.month
    df["week_of_month"] = (df["day_of_month"] - 1) // 7 + 1
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_holiday"] = df["Date"].isin(pd.to_datetime(["2025-05-26", "2025-06-19"])).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df["IntervalIdx"] / 48)
    df["hour_cos"] = np.cos(2 * np.pi * df["IntervalIdx"] / 48)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    df = df.sort_values(["Date", "IntervalIdx"]).reset_index(drop=True)

    df["daily_cv"] = df.groupby("Date")["Call Volume"].transform("sum")

    avg_interval = df.groupby(["day_of_week", "IntervalIdx"])["Call Volume"].mean()
    avg_daily = df.groupby("day_of_week")["Call Volume"].mean()

    def get_hist_share(row):
        denom = max(avg_daily.loc[row["day_of_week"]], 1.0)
        return avg_interval.loc[(row["day_of_week"], row["IntervalIdx"])] / denom

    df["hist_interval_share"] = df.apply(get_hist_share, axis=1)

    df["cv_lag_1"] = df["Call Volume"].shift(1)
    df["cv_lag_2"] = df["Call Volume"].shift(2)
    df["cv_lag_48"] = df["Call Volume"].shift(48)
    df["cv_lag_336"] = df["Call Volume"].shift(48 * 7)

    df["cv_roll_3"] = df["Call Volume"].shift(1).rolling(3).mean()
    df["cv_roll_6"] = df["Call Volume"].shift(1).rolling(6).mean()
    df["cv_roll_12"] = df["Call Volume"].shift(1).rolling(12).mean()

    daily_cv_map = df.groupby("Date")["daily_cv"].first().sort_index()
    daily_cv_lag_map = daily_cv_map.shift(1)

    df["daily_cv_lag_1"] = df["Date"].map(daily_cv_lag_map)

    lag_cols = [
        "cv_lag_1", "cv_lag_2", "cv_lag_48", "cv_lag_336",
        "cv_roll_3", "cv_roll_6", "cv_roll_12",
        "daily_cv_lag_1", "hist_interval_share"
    ]

    for col in lag_cols:
        df[col] = df[col].fillna(df[col].median())

    return df

def build_daily_anchor(daily_df, august_year=2026, upward_bias=1.06):
    d = daily_df.copy().sort_values("Date").reset_index(drop=True)

    d["trend"] = np.arange(len(d))
    d["cv_lag_1"] = d["Call Volume"].shift(1)
    d["cv_lag_7"] = d["Call Volume"].shift(7)
    d["cv_roll_7"] = d["Call Volume"].shift(1).rolling(7).mean()
    d["cv_roll_14"] = d["Call Volume"].shift(1).rolling(14).mean()

    for col in ["cv_lag_1", "cv_lag_7", "cv_roll_7", "cv_roll_14"]:
        d[col] = d[col].fillna(d[col].median())

    train = d.dropna(subset=DAILY_FEATURES + ["Call Volume"]).copy()

    model = HistGradientBoostingRegressor(
        max_iter=350,
        learning_rate=0.04,
        max_depth=5,
        min_samples_leaf=10,
        random_state=42
    )
    model.fit(train[DAILY_FEATURES], train["Call Volume"])

    aug = pd.DataFrame({"Date": pd.date_range(f"{august_year}-08-01", f"{august_year}-08-31", freq="D")})
    aug["day_of_week"] = aug["Date"].dt.dayofweek
    aug["day_of_month"] = aug["Date"].dt.day
    aug["month"] = 8
    aug["week_of_month"] = (aug["day_of_month"] - 1) // 7 + 1
    aug["is_weekend"] = (aug["day_of_week"] >= 5).astype(int)

    last_trend = d["trend"].iloc[-1]
    aug["trend"] = np.arange(last_trend + 1, last_trend + 1 + len(aug))

    recent_cv = d["Call Volume"].tail(21).tolist()
    preds = []

    for i in range(len(aug)):
        cv_lag_1 = recent_cv[-1]
        cv_lag_7 = recent_cv[-7] if len(recent_cv) >= 7 else np.median(recent_cv)
        cv_roll_7 = np.mean(recent_cv[-7:])
        cv_roll_14 = np.mean(recent_cv[-14:]) if len(recent_cv) >= 14 else np.mean(recent_cv)

        row = aug.iloc[[i]].copy()
        row["cv_lag_1"] = cv_lag_1
        row["cv_lag_7"] = cv_lag_7
        row["cv_roll_7"] = cv_roll_7
        row["cv_roll_14"] = cv_roll_14

        pred = max(0, model.predict(row[DAILY_FEATURES])[0])
        preds.append(pred)
        recent_cv.append(pred)

    aug["pred_cv"] = np.array(preds) * upward_bias
    return aug, model

def build_profiles(iv):
    iv_weighted = pd.concat([iv, iv[iv["month"] == 6]], ignore_index=True)

    iv_weighted["cv_share"] = iv_weighted["Call Volume"] / iv_weighted["daily_cv"].clip(lower=1)
    profile_cv = iv_weighted.groupby(["day_of_week", "IntervalIdx"])["cv_share"].mean()

    dow_sums = profile_cv.groupby(level="day_of_week").transform("sum")
    profile_cv = profile_cv / dow_sums.clip(lower=1e-9)

    profile_cct = iv_weighted.groupby(["day_of_week", "IntervalIdx"])["CCT"].mean()
    profile_abd = iv_weighted.groupby(["day_of_week", "IntervalIdx"])["Abandoned Rate"].mean()

    return profile_cv, profile_cct, profile_abd

def train_interval_models(iv):
    train_iv = iv[iv["month"].isin([4, 5])].dropna(subset=INTERVAL_FEATURES + ["Call Volume", "CCT", "Abandoned Rate"])
    val_iv = iv[iv["month"] == 6].dropna(subset=INTERVAL_FEATURES + ["Call Volume", "CCT", "Abandoned Rate"])

    cv_model = HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.03,
        max_depth=6,
        min_samples_leaf=15,
        random_state=42
    )
    cct_model = HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=5,
        min_samples_leaf=20,
        random_state=42
    )
    abd_model = HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=5,
        min_samples_leaf=20,
        random_state=42
    )

    cv_model.fit(train_iv[INTERVAL_FEATURES], train_iv["Call Volume"])
    cct_model.fit(train_iv[INTERVAL_FEATURES], train_iv["CCT"])
    abd_model.fit(train_iv[INTERVAL_FEATURES], train_iv["Abandoned Rate"])

    cv_val_preds = np.maximum(0, cv_model.predict(val_iv[INTERVAL_FEATURES]))
    cct_val_preds = np.maximum(0, cct_model.predict(val_iv[INTERVAL_FEATURES]))
    abd_val_preds = np.clip(abd_model.predict(val_iv[INTERVAL_FEATURES]), 0, 1)

    print(f"    CV  Val MAPE: {safe_mape(val_iv['Call Volume'].values, cv_val_preds):.2f}%")
    print(f"    CCT Val MAPE: {safe_mape(val_iv['CCT'].values, cct_val_preds):.2f}%")
    print(f"    ABD Val MAPE: {safe_mape(val_iv['Abandoned Rate'].values, abd_val_preds):.2f}%")

    return cv_model, cct_model, abd_model

def generate_august_forecast(portfolio, daily_df, interval_df, upward_bias=1.06, cv_blend_weight=0.70):
    print(f"\n{'=' * 60}")
    print(f"Portfolio {portfolio}")
    print(f"{'=' * 60}")

    aug, daily_model = build_daily_anchor(daily_df, august_year=2026, upward_bias=upward_bias)
    print("  Built daily August anchor")

    profile_cv, profile_cct, profile_abd = build_profiles(interval_df)
    print("  Built intraday profiles")

    cv_model, cct_model, abd_model = train_interval_models(interval_df)
    print("  Trained interval models")

    aug["pred_cct"] = aug["day_of_week"].map(daily_df.groupby("day_of_week")["CCT"].mean())
    aug["pred_abd"] = aug["day_of_week"].map(daily_df.groupby("day_of_week")["Abandon Rate"].mean())
    aug["is_holiday"] = 0

    rows = []
    prev_day_total = daily_df["Call Volume"].iloc[-1]
    recent_interval_vals = interval_df["Call Volume"].tail(48 * 7).tolist()

    for _, day_row in aug.iterrows():
        dow = int(day_row["day_of_week"])
        dom = int(day_row["day_of_month"])
        wom = int(day_row["week_of_month"])
        target_daily_cv = float(day_row["pred_cv"])

        iods = np.arange(48)
        feat = pd.DataFrame({
            "IntervalIdx": iods,
            "day_of_week": dow,
            "day_of_month": dom,
            "month": 8,
            "week_of_month": wom,
            "is_weekend": int(dow >= 5),
            "is_holiday": 0,
            "hour_sin": np.sin(2 * np.pi * iods / 48),
            "hour_cos": np.cos(2 * np.pi * iods / 48),
            "dow_sin": np.sin(2 * np.pi * dow / 7),
            "dow_cos": np.cos(2 * np.pi * dow / 7),
            "daily_cv": target_daily_cv,
            "daily_cv_lag_1": prev_day_total,
            "hist_interval_share": [profile_cv.get((dow, iod), 1 / 48) for iod in iods]
        })

        local_recent = recent_interval_vals.copy()
        cv_preds_seq = []

        for iod in iods:
            cv_lag_1 = local_recent[-1] if len(local_recent) >= 1 else target_daily_cv / 48
            cv_lag_2 = local_recent[-2] if len(local_recent) >= 2 else cv_lag_1
            cv_lag_48 = local_recent[-48] if len(local_recent) >= 48 else target_daily_cv / 48
            cv_lag_336 = local_recent[-336] if len(local_recent) >= 336 else target_daily_cv / 48

            recent_for_roll = local_recent[-12:] if len(local_recent) >= 12 else local_recent
            base_val = target_daily_cv / 48 if len(recent_for_roll) == 0 else np.mean(recent_for_roll)

            cv_roll_3 = np.mean(local_recent[-3:]) if len(local_recent) >= 3 else base_val
            cv_roll_6 = np.mean(local_recent[-6:]) if len(local_recent) >= 6 else base_val
            cv_roll_12 = np.mean(local_recent[-12:]) if len(local_recent) >= 12 else base_val

            feat_row = feat.iloc[[iod]].copy()
            feat_row["cv_lag_1"] = cv_lag_1
            feat_row["cv_lag_2"] = cv_lag_2
            feat_row["cv_lag_48"] = cv_lag_48
            feat_row["cv_lag_336"] = cv_lag_336
            feat_row["cv_roll_3"] = cv_roll_3
            feat_row["cv_roll_6"] = cv_roll_6
            feat_row["cv_roll_12"] = cv_roll_12

            model_pred = max(0, cv_model.predict(feat_row[INTERVAL_FEATURES])[0])
            profile_pred = feat_row["hist_interval_share"].iloc[0] * target_daily_cv

            pred = cv_blend_weight * model_pred + (1 - cv_blend_weight) * profile_pred
            pred = max(0, pred)

            cv_preds_seq.append(pred)
            local_recent.append(pred)

        cv_preds_seq = np.array(cv_preds_seq)
        cv_sum = cv_preds_seq.sum()
        if cv_sum > 0:
            cv_preds_seq = cv_preds_seq * (target_daily_cv / cv_sum)

        for iod in iods:
            cv_lag_1 = cv_preds_seq[iod - 1] if iod >= 1 else cv_preds_seq[iod]
            cv_lag_2 = cv_preds_seq[iod - 2] if iod >= 2 else cv_lag_1
            cv_lag_48 = recent_interval_vals[-48 + iod] if len(recent_interval_vals) >= 48 else target_daily_cv / 48
            cv_lag_336 = recent_interval_vals[-336 + iod] if len(recent_interval_vals) >= 336 else target_daily_cv / 48

            if iod >= 1:
                prior_today = cv_preds_seq[:iod]
                cv_roll_3 = prior_today[-3:].mean() if len(prior_today) >= 3 else prior_today.mean()
                cv_roll_6 = prior_today[-6:].mean() if len(prior_today) >= 6 else prior_today.mean()
                cv_roll_12 = prior_today[-12:].mean() if len(prior_today) >= 12 else prior_today.mean()
            else:
                cv_roll_3 = cv_roll_6 = cv_roll_12 = target_daily_cv / 48

            feat_row = pd.DataFrame({
                "IntervalIdx": [iod],
                "day_of_week": [dow],
                "day_of_month": [dom],
                "month": [8],
                "week_of_month": [wom],
                "is_weekend": [int(dow >= 5)],
                "is_holiday": [0],
                "hour_sin": [np.sin(2 * np.pi * iod / 48)],
                "hour_cos": [np.cos(2 * np.pi * iod / 48)],
                "dow_sin": [np.sin(2 * np.pi * dow / 7)],
                "dow_cos": [np.cos(2 * np.pi * dow / 7)],
                "daily_cv": [target_daily_cv],
                "cv_lag_1": [cv_lag_1],
                "cv_lag_2": [cv_lag_2],
                "cv_lag_48": [cv_lag_48],
                "cv_lag_336": [cv_lag_336],
                "cv_roll_3": [cv_roll_3],
                "cv_roll_6": [cv_roll_6],
                "cv_roll_12": [cv_roll_12],
                "daily_cv_lag_1": [prev_day_total],
                "hist_interval_share": [profile_cv.get((dow, iod), 1 / 48)]
            })

            cv = float(max(0, cv_preds_seq[iod]))

            cct_model_pred = max(0, cct_model.predict(feat_row[INTERVAL_FEATURES])[0])
            cct_profile_pred = profile_cct.get((dow, iod), day_row["pred_cct"])
            cct = 0.7 * cct_model_pred + 0.3 * cct_profile_pred
            cct = max(0, cct)

            abd_model_pred = np.clip(abd_model.predict(feat_row[INTERVAL_FEATURES])[0], 0, 1)
            abd_profile_pred = profile_abd.get((dow, iod), day_row["pred_abd"])
            abd = 0.7 * abd_model_pred + 0.3 * abd_profile_pred
            abd = float(np.clip(abd, 0, 1))

            h = iod // 2
            mn = (iod % 2) * 30

            rows.append({
                "Month": "August",
                "Day": dom,
                "Interval": f"{h}:{mn:02d}",
                f"Calls_Offered_{portfolio}": round(cv, 2),
                f"Abandoned_Calls_{portfolio}": round(cv * abd, 2),
                f"Abandoned_Rate_{portfolio}": round(abd, 6),
                f"CCT_{portfolio}": round(cct, 2)
            })

        prev_day_total = target_daily_cv
        recent_interval_vals.extend(cv_preds_seq.tolist())
        recent_interval_vals = recent_interval_vals[-(48 * 14):]

    forecast_df = pd.DataFrame(rows)
    print(f"  Generated {len(forecast_df)} rows")
    return forecast_df

all_forecasts = []

for p in PORTFOLIOS:
    daily_df = load_daily(p)
    interval_df = load_interval(p)
    fcst = generate_august_forecast(
        portfolio=p,
        daily_df=daily_df,
        interval_df=interval_df,
        upward_bias=1.06,
        cv_blend_weight=0.70
    )
    all_forecasts.append(fcst)

print("\nCombining portfolios...")
result = all_forecasts[0][[
    "Month", "Day", "Interval",
    "Calls_Offered_A", "Abandoned_Calls_A", "Abandoned_Rate_A", "CCT_A"
]]

for i, p in enumerate(["B", "C", "D"]):
    result = result.merge(
        all_forecasts[i + 1][[
            "Month", "Day", "Interval",
            f"Calls_Offered_{p}", f"Abandoned_Calls_{p}",
            f"Abandoned_Rate_{p}", f"CCT_{p}"
        ]],
        on=["Month", "Day", "Interval"],
        how="left"
    )

col_order = ["Month", "Day", "Interval"]
for p in PORTFOLIOS:
    col_order += [
        f"Calls_Offered_{p}",
        f"Abandoned_Calls_{p}",
        f"Abandoned_Rate_{p}",
        f"CCT_{p}"
    ]

result = result[col_order]

for p in PORTFOLIOS:
    result[f"Calls_Offered_{p}"] = result[f"Calls_Offered_{p}"].clip(lower=0)
    result[f"Abandoned_Calls_{p}"] = result[f"Abandoned_Calls_{p}"].clip(lower=0)
    result[f"Abandoned_Rate_{p}"] = result[f"Abandoned_Rate_{p}"].clip(0, 1)
    result[f"CCT_{p}"] = result[f"CCT_{p}"].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved to {OUTPUT_PATH}")
print(f"Shape: {result.shape}")
print(f"Negatives present: {(result.select_dtypes(include=[np.number]) < 0).any().any()}")
print(f"Nulls present: {result.isnull().any().any()}")

print("\nDaily totals by portfolio:")
for p in PORTFOLIOS:
    daily_totals = result.groupby("Day")[f"Calls_Offered_{p}"].sum()
    print(f"  {p}: mean={daily_totals.mean():.0f}, min={daily_totals.min():.0f}, max={daily_totals.max():.0f}")

print("\nFirst 5 rows:")
print(result.head().to_string(index=False))


Portfolio A
  Built daily August anchor
  Built intraday profiles
    CV  Val MAPE: 27.03%
    CCT Val MAPE: 28.62%
    ABD Val MAPE: 58.35%
  Trained interval models
  Generated 1488 rows

Portfolio B
  Built daily August anchor
  Built intraday profiles
    CV  Val MAPE: 20.84%
    CCT Val MAPE: 20.69%
    ABD Val MAPE: 114.47%
  Trained interval models
  Generated 1488 rows

Portfolio C
  Built daily August anchor
  Built intraday profiles
    CV  Val MAPE: 11.65%
    CCT Val MAPE: 9.50%
    ABD Val MAPE: 190.31%
  Trained interval models
  Generated 1488 rows

Portfolio D
  Built daily August anchor
  Built intraday profiles
    CV  Val MAPE: 14.82%
    CCT Val MAPE: 19.49%
    ABD Val MAPE: 141.74%
  Trained interval models
  Generated 1488 rows

Combining portfolios...

Saved to ./forecast_v07.csv
Shape: (1488, 19)
Negatives present: False
Nulls present: False

Daily totals by portfolio:
  A: mean=3530, min=1067, max=5634
  B: mean=8052, min=4075, max=11504
  C: mean=18126, min=

In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

PORTFOLIOS  = ['A', 'B', 'C', 'D']
DATA_DIR    = './data'
OUTPUT_PATH = './forecast_v08.csv'

DAILY_FEATURES = [
    'day_of_week', 'month', 'day_of_month', 'week_of_month', 'is_weekend',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'trend',
    'cv_lag_1', 'cv_lag_7', 'cv_roll_7', 'cv_roll_14',
    'cct_lag_1', 'cct_lag_7', 'cct_roll_7',
    'abd_lag_1', 'abd_lag_7', 'abd_roll_7',
    'cv_lag_364', 'cct_lag_364', 'abd_lag_364',
    'aug_cv_mean', 'aug_cct_mean', 'aug_abd_mean',
    'aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean'
]

def safe_mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true > 0
    if mask.sum() == 0:
        return np.nan
    return mean_absolute_percentage_error(y_true[mask], y_pred[mask]) * 100

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN INTERVAL DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_interval(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()

    df['Interval'] = df['Interval'].apply(
        lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x)
    )

    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })

    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = (
            df.groupby(['Month', 'Day'])[col]
              .transform(lambda x: x.interpolate(method='linear').bfill().ffill())
        )

    df['Date']        = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['IntervalIdx'] = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )

    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df.groupby(['day_of_week', 'IntervalIdx'])[col].transform(
            lambda x: x.fillna(x.median())
        )

    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)

    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    df['day_of_month']  = df['Date'].dt.day
    df['month']         = df['Date'].dt.month
    df['week_of_month'] = (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']    = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday']    = df['Date'].isin(pd.to_datetime(['2025-05-26', '2025-06-19'])).astype(int)
    df['hour_sin']      = np.sin(2 * np.pi * df['IntervalIdx'] / 48)
    df['hour_cos']      = np.cos(2 * np.pi * df['IntervalIdx'] / 48)
    df['dow_sin']       = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']       = np.cos(2 * np.pi * df['day_of_week'] / 7)

    remaining = df[['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']].isnull().sum().sum()
    print(f'  {p} interval nulls after cleanup: {remaining}')
    return df

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN DAILY DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_daily(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()

    df['Date'] = pd.to_datetime(df['Date'].astype(str).str[:8], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)

    df['day_of_week']  = df['Date'].dt.dayofweek
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('month')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df[col].fillna(df[col].median())

    df[['Call Volume', 'CCT', 'Abandon Rate']] = df[['Call Volume', 'CCT', 'Abandon Rate']].clip(lower=0)

    df['dow_sin']   = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['trend']     = np.arange(len(df))

    df['cv_lag_1']   = df['Call Volume'].shift(1)
    df['cv_lag_7']   = df['Call Volume'].shift(7)
    df['cv_roll_7']  = df['Call Volume'].shift(1).rolling(7).mean()
    df['cv_roll_14'] = df['Call Volume'].shift(1).rolling(14).mean()

    df['cct_lag_1']  = df['CCT'].shift(1)
    df['cct_lag_7']  = df['CCT'].shift(7)
    df['cct_roll_7'] = df['CCT'].shift(1).rolling(7).mean()

    df['abd_lag_1']  = df['Abandon Rate'].shift(1)
    df['abd_lag_7']  = df['Abandon Rate'].shift(7)
    df['abd_roll_7'] = df['Abandon Rate'].shift(1).rolling(7).mean()

    df['cv_lag_364']  = df['Call Volume'].shift(364)
    df['cct_lag_364'] = df['CCT'].shift(364)
    df['abd_lag_364'] = df['Abandon Rate'].shift(364)

    aug_data = df[df['month'] == 8].copy()
    df['aug_cv_mean']  = aug_data['Call Volume'].mean()
    df['aug_cct_mean'] = aug_data['CCT'].mean()
    df['aug_abd_mean'] = aug_data['Abandon Rate'].mean()

    aug_dow = aug_data.groupby('day_of_week')[['Call Volume','CCT','Abandon Rate']].mean()
    aug_dow.columns = ['aug_dow_cv_mean','aug_dow_cct_mean','aug_dow_abd_mean']
    df = df.merge(aug_dow, on='day_of_week', how='left')

    fill_cols = [
        'cv_lag_1', 'cv_lag_7', 'cv_roll_7', 'cv_roll_14',
        'cct_lag_1', 'cct_lag_7', 'cct_roll_7',
        'abd_lag_1', 'abd_lag_7', 'abd_roll_7',
        'cv_lag_364', 'cct_lag_364', 'abd_lag_364',
        'aug_cv_mean', 'aug_cct_mean', 'aug_abd_mean',
        'aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean'
    ]
    for col in fill_cols:
        df[col] = df[col].fillna(df[col].median())

    remaining = df[['Call Volume', 'CCT', 'Abandon Rate']].isnull().sum().sum()
    print(f'  {p} daily nulls after cleanup: {remaining}')
    return df

# ══════════════════════════════════════════════════════════════════════════════
# DAILY MODELING
# ══════════════════════════════════════════════════════════════════════════════
def train_daily_models(d):
    train_d = d.dropna(subset=DAILY_FEATURES + ['Call Volume', 'CCT', 'Abandon Rate']).copy()
    print(f'Daily training rows: {len(train_d)}')

    daily_models = {}
    for target in ['Call Volume', 'CCT', 'Abandon Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=350,
            learning_rate=0.04,
            max_depth=5,
            min_samples_leaf=10,
            random_state=42
        )
        m.fit(train_d[DAILY_FEATURES], train_d[target])
        daily_models[target] = m

    return daily_models

def build_august_daily_grid(d, daily_models, bias_mult=1.08):
    aug_days = pd.date_range('2026-08-01', '2026-08-31', freq='D')

    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'month':         aug_days.month,
        'day_of_month':  aug_days.day,
        'week_of_month': (aug_days.day - 1) // 7 + 1,
        'is_weekend':    (aug_days.dayofweek >= 5).astype(int),
    })
    aug['dow_sin']   = np.sin(2 * np.pi * aug['day_of_week'] / 7)
    aug['dow_cos']   = np.cos(2 * np.pi * aug['day_of_week'] / 7)
    aug['month_sin'] = np.sin(2 * np.pi * aug['month'] / 12)
    aug['month_cos'] = np.cos(2 * np.pi * aug['month'] / 12)

    last_trend = d['trend'].iloc[-1]
    aug['trend'] = np.arange(last_trend + 1, last_trend + 1 + len(aug))

    aug_2025 = d[(d['Date'].dt.month == 8) & (d['Date'].dt.year == 2025)].reset_index(drop=True)
    aug_all  = d[d['month'] == 8].copy()

    aug['aug_cv_mean']  = aug_all['Call Volume'].mean()
    aug['aug_cct_mean'] = aug_all['CCT'].mean()
    aug['aug_abd_mean'] = aug_all['Abandon Rate'].mean()

    aug_dow_means = aug_all.groupby('day_of_week')[['Call Volume','CCT','Abandon Rate']].mean()
    aug_dow_means.columns = ['aug_dow_cv_mean','aug_dow_cct_mean','aug_dow_abd_mean']
    aug = aug.merge(aug_dow_means, on='day_of_week', how='left')

    if len(aug_2025) == len(aug):
        aug['cv_lag_364']  = aug_2025['Call Volume'].values
        aug['cct_lag_364'] = aug_2025['CCT'].values
        aug['abd_lag_364'] = aug_2025['Abandon Rate'].values
    else:
        aug['cv_lag_364']  = d['Call Volume'].tail(31).mean()
        aug['cct_lag_364'] = d['CCT'].tail(31).mean()
        aug['abd_lag_364'] = d['Abandon Rate'].tail(31).mean()

    recent_cv  = d['Call Volume'].tail(21).tolist()
    recent_cct = d['CCT'].tail(21).tolist()
    recent_abd = d['Abandon Rate'].tail(21).tolist()

    pred_cv = []
    pred_cct = []
    pred_abd = []

    for i in range(len(aug)):
        row = aug.iloc[[i]].copy()

        row['cv_lag_1']   = recent_cv[-1]
        row['cv_lag_7']   = recent_cv[-7] if len(recent_cv) >= 7 else np.median(recent_cv)
        row['cv_roll_7']  = np.mean(recent_cv[-7:])
        row['cv_roll_14'] = np.mean(recent_cv[-14:]) if len(recent_cv) >= 14 else np.mean(recent_cv)

        row['cct_lag_1']  = recent_cct[-1]
        row['cct_lag_7']  = recent_cct[-7] if len(recent_cct) >= 7 else np.median(recent_cct)
        row['cct_roll_7'] = np.mean(recent_cct[-7:])

        row['abd_lag_1']  = recent_abd[-1]
        row['abd_lag_7']  = recent_abd[-7] if len(recent_abd) >= 7 else np.median(recent_abd)
        row['abd_roll_7'] = np.mean(recent_abd[-7:])

        cv  = max(0, daily_models['Call Volume'].predict(row[DAILY_FEATURES])[0])
        cct = max(0, daily_models['CCT'].predict(row[DAILY_FEATURES])[0])
        abd = float(np.clip(daily_models['Abandon Rate'].predict(row[DAILY_FEATURES])[0], 0, 1))

        pred_cv.append(cv)
        pred_cct.append(cct)
        pred_abd.append(abd)

        recent_cv.append(cv)
        recent_cct.append(cct)
        recent_abd.append(abd)

    aug['pred_cv']  = np.array(pred_cv) * bias_mult
    aug['pred_cct'] = np.array(pred_cct)
    aug['pred_abd'] = np.array(pred_abd)

    return aug

# ══════════════════════════════════════════════════════════════════════════════
# INTRADAY PROFILES
# ══════════════════════════════════════════════════════════════════════════════
def build_profiles(iv):
    iv = iv.copy()
    iv['daily_cv'] = iv.groupby('Date')['Call Volume'].transform('sum')
    iv['cv_share'] = iv['Call Volume'] / iv['daily_cv'].clip(lower=1)

    iv_weighted = pd.concat([iv, iv[iv['month'] == 6], iv[iv['month'] == 6]], ignore_index=True)

    profile_cv_dow = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
    profile_cv_all = iv_weighted.groupby('IntervalIdx')['cv_share'].mean()
    profile_cv_wk  = iv_weighted.groupby(['is_weekend', 'IntervalIdx'])['cv_share'].mean()

    profile_cct_dow = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean()
    profile_cct_all = iv_weighted.groupby('IntervalIdx')['CCT'].mean()
    profile_cct_wk  = iv_weighted.groupby(['is_weekend', 'IntervalIdx'])['CCT'].mean()

    profile_abd_dow = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean()
    profile_abd_all = iv_weighted.groupby('IntervalIdx')['Abandoned Rate'].mean()
    profile_abd_wk  = iv_weighted.groupby(['is_weekend', 'IntervalIdx'])['Abandoned Rate'].mean()

    smooth_cv = {}
    smooth_cct = {}
    smooth_abd = {}

    for dow in range(7):
        is_weekend = int(dow >= 5)
        shares = []
        for iod in range(48):
            s1 = profile_cv_dow.get((dow, iod), np.nan)
            s2 = profile_cv_wk.get((is_weekend, iod), np.nan)
            s3 = profile_cv_all.get(iod, 1/48)
            vals = [v for v in [s1, s2, s3] if pd.notna(v)]
            if len(vals) == 3:
                share = 0.60 * vals[0] + 0.25 * vals[1] + 0.15 * vals[2]
            elif len(vals) == 2:
                share = 0.75 * vals[0] + 0.25 * vals[1]
            else:
                share = vals[0] if len(vals) else 1/48
            shares.append(max(0, share))

        shares = np.array(shares)
        shares = shares / shares.sum().clip(min=1e-9)

        for iod in range(48):
            smooth_cv[(dow, iod)] = shares[iod]

            c1 = profile_cct_dow.get((dow, iod), np.nan)
            c2 = profile_cct_wk.get((is_weekend, iod), np.nan)
            c3 = profile_cct_all.get(iod, np.nan)
            cvals = [v for v in [c1, c2, c3] if pd.notna(v)]
            if len(cvals) == 3:
                smooth_cct[(dow, iod)] = max(0, 0.60 * cvals[0] + 0.25 * cvals[1] + 0.15 * cvals[2])
            elif len(cvals) == 2:
                smooth_cct[(dow, iod)] = max(0, 0.75 * cvals[0] + 0.25 * cvals[1])
            elif len(cvals) == 1:
                smooth_cct[(dow, iod)] = max(0, cvals[0])
            else:
                smooth_cct[(dow, iod)] = 0

            a1 = profile_abd_dow.get((dow, iod), np.nan)
            a2 = profile_abd_wk.get((is_weekend, iod), np.nan)
            a3 = profile_abd_all.get(iod, np.nan)
            avals = [v for v in [a1, a2, a3] if pd.notna(v)]
            if len(avals) == 3:
                smooth_abd[(dow, iod)] = float(np.clip(0.60 * avals[0] + 0.25 * avals[1] + 0.15 * avals[2], 0, 1))
            elif len(avals) == 2:
                smooth_abd[(dow, iod)] = float(np.clip(0.75 * avals[0] + 0.25 * avals[1], 0, 1))
            elif len(avals) == 1:
                smooth_abd[(dow, iod)] = float(np.clip(avals[0], 0, 1))
            else:
                smooth_abd[(dow, iod)] = 0.01

    return smooth_cv, smooth_cct, smooth_abd

def calibrate_profiles_on_june(iv, profile_cv, profile_cct, profile_abd):
    june = iv[iv['month'] == 6].copy()
    june['actual_daily'] = june.groupby('Date')['Call Volume'].transform('sum')

    june['cv_share_pred'] = june.apply(lambda r: profile_cv.get((r['day_of_week'], r['IntervalIdx']), 1/48), axis=1)
    june['cv_pred'] = june['cv_share_pred'] * june['actual_daily']

    june['cct_pred'] = june.apply(lambda r: profile_cct.get((r['day_of_week'], r['IntervalIdx']), np.nan), axis=1)
    june['abd_pred'] = june.apply(lambda r: profile_abd.get((r['day_of_week'], r['IntervalIdx']), np.nan), axis=1)

    cv_cal = june.groupby('IntervalIdx').apply(
        lambda x: x['Call Volume'].sum() / max(x['cv_pred'].sum(), 1e-9)
    ).clip(0.85, 1.15).to_dict()

    cct_cal = june.groupby('IntervalIdx').apply(
        lambda x: x['CCT'].mean() / max(x['cct_pred'].mean(), 1e-9)
    ).clip(0.85, 1.15).to_dict()

    abd_cal = june.groupby('IntervalIdx').apply(
        lambda x: x['Abandoned Rate'].mean() / max(x['abd_pred'].mean(), 1e-9)
    ).clip(0.75, 1.25).to_dict()

    mask_cv = june['Call Volume'] > 0
    if mask_cv.sum() > 0:
        mape_cv = safe_mape(june.loc[mask_cv, 'Call Volume'], june.loc[mask_cv, 'cv_pred'])
        print(f'CV  profile MAPE on June (pre-calibration): {mape_cv:.2f}%')

    mask_cct = june['CCT'] > 0
    if mask_cct.sum() > 0:
        mape_cct = safe_mape(june.loc[mask_cct, 'CCT'], june.loc[mask_cct, 'cct_pred'])
        print(f'CCT profile MAPE on June (pre-calibration): {mape_cct:.2f}%')

    mask_abd = june['Abandoned Rate'] > 0
    if mask_abd.sum() > 0:
        mape_abd = safe_mape(june.loc[mask_abd, 'Abandoned Rate'], june.loc[mask_abd, 'abd_pred'])
        print(f'ABD profile MAPE on June (pre-calibration): {mape_abd:.2f}%')

    return cv_cal, cct_cal, abd_cal

# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
all_forecasts = []

for p in PORTFOLIOS:
    print(f"\n{'='*60}")
    print(f'Portfolio {p}')
    print(f"{'='*60}")

    d  = load_daily(p)
    iv = load_interval(p)

    daily_models = train_daily_models(d)
    aug = build_august_daily_grid(d, daily_models, bias_mult=1.08)

    print('August daily predictions:')
    print(aug[['Date', 'pred_cv', 'pred_cct', 'pred_abd']].to_string(index=False))

    profile_cv, profile_cct, profile_abd = build_profiles(iv)
    cv_cal, cct_cal, abd_cal = calibrate_profiles_on_june(iv, profile_cv, profile_cct, profile_abd)

    rows = []
    for _, day_row in aug.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])

        cv_preds = []
        cct_preds = []
        abd_preds = []

        for iod in range(48):
            base_cv_share = profile_cv.get((dow, iod), 1/48)
            cal_cv_share  = base_cv_share * cv_cal.get(iod, 1.0)
            cv_preds.append(max(0, cal_cv_share * day_row['pred_cv']))

            base_cct = profile_cct.get((dow, iod), day_row['pred_cct'])
            cct_preds.append(max(0, base_cct * cct_cal.get(iod, 1.0)))

            base_abd = profile_abd.get((dow, iod), day_row['pred_abd'])
            abd_preds.append(float(np.clip(base_abd * abd_cal.get(iod, 1.0), 0, 1)))

        cv_preds = np.array(cv_preds)
        if cv_preds.sum() > 0:
            cv_preds = cv_preds * (day_row['pred_cv'] / cv_preds.sum())

        for iod in range(48):
            h  = iod // 2
            mn = (iod % 2) * 30

            cv_pred  = float(max(0, cv_preds[iod]))
            cct_pred = float(max(0, cct_preds[iod]))
            abd_pred = float(np.clip(abd_preds[iod], 0, 1))

            rows.append({
                'Month':                 'August',
                'Day':                   dom,
                'Interval':              f'{h}:{mn:02d}',
                f'Calls_Offered_{p}':    round(cv_pred, 2),
                f'Abandoned_Calls_{p}':  round(cv_pred * abd_pred, 2),
                f'Abandoned_Rate_{p}':   round(abd_pred, 6),
                f'CCT_{p}':              round(cct_pred, 2),
            })

    out = pd.DataFrame(rows)
    all_forecasts.append(out)
    print(f'{p}: {len(out)} rows generated')

# ══════════════════════════════════════════════════════════════════════════════
# COMBINE
# ══════════════════════════════════════════════════════════════════════════════
print('\nCombining portfolios...')
result = all_forecasts[0][[
    'Month', 'Day', 'Interval',
    'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A'
]]

for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(
        all_forecasts[i + 1][[
            'Month', 'Day', 'Interval',
            f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
            f'Abandoned_Rate_{p}', f'CCT_{p}'
        ]],
        on=['Month', 'Day', 'Interval'], how='left'
    )

col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]

result = result[col_order]

for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n✅ Saved to {OUTPUT_PATH}')
print(f'   Shape:     {result.shape}  |  Expected: (1488, 19)')
print(f'   Negatives: {(result.select_dtypes("number") < 0).any().any()}')
print(f'   Nulls:     {result.isnull().any().any()}')

print('\n=== Daily CV totals by portfolio ===')
for p in PORTFOLIOS:
    col   = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f'  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}')

print('\n=== First 5 rows ===')
print(result.head(5).to_string(index=False))


Portfolio A
  A daily nulls after cleanup: 0
  A interval nulls after cleanup: 0
Daily training rows: 731
August daily predictions:
      Date     pred_cv   pred_cct  pred_abd
2026-08-01 2950.896567 322.763872  0.017311
2026-08-02 1679.471517 322.964015  0.014960
2026-08-03 4478.807070 298.960357  0.015451
2026-08-04 3950.695358 310.782791  0.018074
2026-08-05 4077.147435 306.363229  0.011253
2026-08-06 3536.350067 308.431716  0.011990
2026-08-07 3868.214430 314.076535  0.009215
2026-08-08 2970.941515 306.628645  0.009606
2026-08-09 1264.822265 303.380234  0.011742
2026-08-10 3897.255225 305.174826  0.016883
2026-08-11 3943.674103 307.397526  0.012511
2026-08-12 3580.275326 303.641764  0.010577
2026-08-13 3612.027581 309.636391  0.011758
2026-08-14 3540.984314 312.918067  0.014019
2026-08-15 3059.596357 306.427913  0.009418
2026-08-16 1324.982932 303.730477  0.011668
2026-08-17 4033.924916 307.814463  0.015026
2026-08-18 3714.965548 307.289814  0.009605
2026-08-19 3580.627778 305.8801

In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

PORTFOLIOS  = ['A', 'B', 'C', 'D']
DATA_DIR    = './data'
OUTPUT_PATH = './forecast_v04_improved.csv'

DAILY_FEATURES = [
    'day_of_week', 'month', 'day_of_month', 'week_of_month', 'is_weekend',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'cv_lag_1', 'cv_lag_7', 'cv_roll_7', 'cv_roll_14',
    'cct_lag_1', 'cct_lag_7', 'cct_roll_7',
    'abd_lag_1', 'abd_lag_7', 'abd_roll_7',
    'cv_lag_364', 'cct_lag_364', 'abd_lag_364',
    'aug_cv_mean', 'aug_cct_mean', 'aug_abd_mean',
    'aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean'
]

def safe_mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true > 0
    if mask.sum() == 0:
        return np.nan
    return mean_absolute_percentage_error(y_true[mask], y_pred[mask]) * 100

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN INTERVAL DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_interval(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()

    df['Interval'] = df['Interval'].apply(
        lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x)
    )

    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({
        'Month':    dr.month_name(),
        'Day':      dr.day,
        'Interval': dr.strftime('%H:%M:%S'),
    })
    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')

    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = (
            df.groupby(['Month', 'Day'])[col]
              .transform(lambda x: x.interpolate(method='linear').bfill().ffill())
        )

    df['Date'] = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['IntervalIdx'] = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )

    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df.groupby(['day_of_week', 'IntervalIdx'])[col].transform(
            lambda x: x.fillna(x.median())
        )

    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)

    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)

    df['day_of_month'] = df['Date'].dt.day
    df['month'] = df['Date'].dt.month
    df['week_of_month'] = (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday'] = df['Date'].isin(pd.to_datetime(['2025-05-26', '2025-06-19'])).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['IntervalIdx'] / 48)
    df['hour_cos'] = np.cos(2 * np.pi * df['IntervalIdx'] / 48)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    remaining = df[['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']].isnull().sum().sum()
    print(f"  {p} interval nulls after cleanup: {remaining}")
    return df

# ══════════════════════════════════════════════════════════════════════════════
# LOAD & CLEAN DAILY DATA
# ══════════════════════════════════════════════════════════════════════════════
def load_daily(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()

    df['Date'] = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)

    df['day_of_week'] = df['Date'].dt.dayofweek
    df['month'] = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month'] = (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('month')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df[col].fillna(df[col].median())

    df[['Call Volume', 'CCT', 'Abandon Rate']] = df[['Call Volume', 'CCT', 'Abandon Rate']].clip(lower=0)

    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    df['cv_lag_1'] = df['Call Volume'].shift(1)
    df['cv_lag_7'] = df['Call Volume'].shift(7)
    df['cv_roll_7'] = df['Call Volume'].shift(1).rolling(7).mean()
    df['cv_roll_14'] = df['Call Volume'].shift(1).rolling(14).mean()

    df['cct_lag_1'] = df['CCT'].shift(1)
    df['cct_lag_7'] = df['CCT'].shift(7)
    df['cct_roll_7'] = df['CCT'].shift(1).rolling(7).mean()

    df['abd_lag_1'] = df['Abandon Rate'].shift(1)
    df['abd_lag_7'] = df['Abandon Rate'].shift(7)
    df['abd_roll_7'] = df['Abandon Rate'].shift(1).rolling(7).mean()

    df['cv_lag_364'] = df['Call Volume'].shift(364)
    df['cct_lag_364'] = df['CCT'].shift(364)
    df['abd_lag_364'] = df['Abandon Rate'].shift(364)

    aug_data = df[df['month'] == 8].copy()
    df['aug_cv_mean'] = aug_data['Call Volume'].mean()
    df['aug_cct_mean'] = aug_data['CCT'].mean()
    df['aug_abd_mean'] = aug_data['Abandon Rate'].mean()

    aug_dow = aug_data.groupby('day_of_week')[['Call Volume', 'CCT', 'Abandon Rate']].mean()
    aug_dow.columns = ['aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean']
    df = df.merge(aug_dow, on='day_of_week', how='left')

    fill_cols = [
        'cv_lag_1', 'cv_lag_7', 'cv_roll_7', 'cv_roll_14',
        'cct_lag_1', 'cct_lag_7', 'cct_roll_7',
        'abd_lag_1', 'abd_lag_7', 'abd_roll_7',
        'cv_lag_364', 'cct_lag_364', 'abd_lag_364',
        'aug_cv_mean', 'aug_cct_mean', 'aug_abd_mean',
        'aug_dow_cv_mean', 'aug_dow_cct_mean', 'aug_dow_abd_mean'
    ]
    for col in fill_cols:
        df[col] = df[col].fillna(df[col].median())

    remaining = df[['Call Volume', 'CCT', 'Abandon Rate']].isnull().sum().sum()
    print(f"  {p} daily nulls after cleanup: {remaining}")
    return df

# ══════════════════════════════════════════════════════════════════════════════
# DAILY MODELS
# ══════════════════════════════════════════════════════════════════════════════
def train_daily_models(d):
    train_d = d.dropna(subset=DAILY_FEATURES + ['Call Volume'])
    print(f"Daily training rows: {len(train_d)}")

    daily_models = {}
    for target in ['Call Volume', 'CCT', 'Abandon Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_depth=5,
            random_state=42
        )
        m.fit(train_d[DAILY_FEATURES], train_d[target])
        daily_models[target] = m

    return daily_models

def build_august_daily_grid(d, daily_models, bias_mult=1.10):
    aug_days = pd.date_range('2026-08-01', '2026-08-31', freq='D')

    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'month':         aug_days.month,
        'day_of_month':  aug_days.day,
        'week_of_month': (aug_days.day - 1) // 7 + 1,
        'is_weekend':    (aug_days.dayofweek >= 5).astype(int),
    })
    aug['dow_sin']   = np.sin(2 * np.pi * aug['day_of_week'] / 7)
    aug['dow_cos']   = np.cos(2 * np.pi * aug['day_of_week'] / 7)
    aug['month_sin'] = np.sin(2 * np.pi * aug['month'] / 12)
    aug['month_cos'] = np.cos(2 * np.pi * aug['month'] / 12)

    aug_2025 = d[(d['Date'].dt.month == 8) & (d['Date'].dt.year == 2025)].reset_index(drop=True)
    aug_all = d[d['month'] == 8]

    aug['aug_cv_mean']  = aug_all['Call Volume'].mean()
    aug['aug_cct_mean'] = aug_all['CCT'].mean()
    aug['aug_abd_mean'] = aug_all['Abandon Rate'].mean()

    aug_dow_means = aug_all.groupby('day_of_week')[['Call Volume','CCT','Abandon Rate']].mean()
    aug_dow_means.columns = ['aug_dow_cv_mean','aug_dow_cct_mean','aug_dow_abd_mean']
    aug = aug.merge(aug_dow_means, on='day_of_week', how='left')

    aug['cv_lag_364']  = aug_2025['Call Volume'].values if len(aug_2025) == 31 else d['Call Volume'].tail(31).mean()
    aug['cct_lag_364'] = aug_2025['CCT'].values if len(aug_2025) == 31 else d['CCT'].tail(31).mean()
    aug['abd_lag_364'] = aug_2025['Abandon Rate'].values if len(aug_2025) == 31 else d['Abandon Rate'].tail(31).mean()

    recent_cv = d['Call Volume'].tail(21).tolist()
    recent_cct = d['CCT'].tail(21).tolist()
    recent_abd = d['Abandon Rate'].tail(21).tolist()

    pred_cv, pred_cct, pred_abd = [], [], []

    for i in range(len(aug)):
        row = aug.iloc[[i]].copy()

        row['cv_lag_1'] = recent_cv[-1]
        row['cv_lag_7'] = recent_cv[-7] if len(recent_cv) >= 7 else np.median(recent_cv)
        row['cv_roll_7'] = np.mean(recent_cv[-7:])
        row['cv_roll_14'] = np.mean(recent_cv[-14:]) if len(recent_cv) >= 14 else np.mean(recent_cv)

        row['cct_lag_1'] = recent_cct[-1]
        row['cct_lag_7'] = recent_cct[-7] if len(recent_cct) >= 7 else np.median(recent_cct)
        row['cct_roll_7'] = np.mean(recent_cct[-7:])

        row['abd_lag_1'] = recent_abd[-1]
        row['abd_lag_7'] = recent_abd[-7] if len(recent_abd) >= 7 else np.median(recent_abd)
        row['abd_roll_7'] = np.mean(recent_abd[-7:])

        cv = max(0, daily_models['Call Volume'].predict(row[DAILY_FEATURES])[0])
        cct = max(0, daily_models['CCT'].predict(row[DAILY_FEATURES])[0])
        abd = float(np.clip(daily_models['Abandon Rate'].predict(row[DAILY_FEATURES])[0], 0, 1))

        pred_cv.append(cv)
        pred_cct.append(cct)
        pred_abd.append(abd)

        recent_cv.append(cv)
        recent_cct.append(cct)
        recent_abd.append(abd)

    aug['pred_cv'] = np.array(pred_cv) * bias_mult
    aug['pred_cct'] = np.array(pred_cct)
    aug['pred_abd'] = np.array(pred_abd)

    return aug

# ══════════════════════════════════════════════════════════════════════════════
# PROFILES
# ══════════════════════════════════════════════════════════════════════════════
def build_profiles(iv):
    iv = iv.copy()
    iv['daily_cv'] = iv.groupby('Date')['Call Volume'].transform('sum')
    iv['cv_share'] = iv['Call Volume'] / iv['daily_cv'].clip(lower=1)

    iv_weighted = pd.concat([iv, iv[iv['month'] == 6], iv[iv['month'] == 6]], ignore_index=True)

    profile_cv_dow = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
    profile_cv_all = iv_weighted.groupby('IntervalIdx')['cv_share'].mean()
    profile_cv_wk = iv_weighted.groupby(['is_weekend', 'IntervalIdx'])['cv_share'].mean()

    profile_cct_dow = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean()
    profile_cct_all = iv_weighted.groupby('IntervalIdx')['CCT'].mean()
    profile_cct_wk = iv_weighted.groupby(['is_weekend', 'IntervalIdx'])['CCT'].mean()

    profile_abd_dow = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean()
    profile_abd_all = iv_weighted.groupby('IntervalIdx')['Abandoned Rate'].mean()
    profile_abd_wk = iv_weighted.groupby(['is_weekend', 'IntervalIdx'])['Abandoned Rate'].mean()

    smooth_cv, smooth_cct, smooth_abd = {}, {}, {}

    for dow in range(7):
        is_weekend = int(dow >= 5)
        shares = []
        for iod in range(48):
            s1 = profile_cv_dow.get((dow, iod), np.nan)
            s2 = profile_cv_wk.get((is_weekend, iod), np.nan)
            s3 = profile_cv_all.get(iod, 1/48)
            vals = [v for v in [s1, s2, s3] if pd.notna(v)]
            if len(vals) == 3:
                share = 0.60 * vals[0] + 0.25 * vals[1] + 0.15 * vals[2]
            elif len(vals) == 2:
                share = 0.75 * vals[0] + 0.25 * vals[1]
            else:
                share = vals[0] if len(vals) else 1/48
            shares.append(max(0, share))

        shares = np.array(shares)
        shares = shares / max(shares.sum(), 1e-9)

        for iod in range(48):
            smooth_cv[(dow, iod)] = shares[iod]

            c1 = profile_cct_dow.get((dow, iod), np.nan)
            c2 = profile_cct_wk.get((is_weekend, iod), np.nan)
            c3 = profile_cct_all.get(iod, np.nan)
            cvals = [v for v in [c1, c2, c3] if pd.notna(v)]
            smooth_cct[(dow, iod)] = max(0, np.mean(cvals)) if len(cvals) else 0

            a1 = profile_abd_dow.get((dow, iod), np.nan)
            a2 = profile_abd_wk.get((is_weekend, iod), np.nan)
            a3 = profile_abd_all.get(iod, np.nan)
            avals = [v for v in [a1, a2, a3] if pd.notna(v)]
            smooth_abd[(dow, iod)] = float(np.clip(np.mean(avals), 0, 1)) if len(avals) else 0.01

    return smooth_cv, smooth_cct, smooth_abd

# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════
all_forecasts = []

for p in PORTFOLIOS:
    print(f"\n{'='*50}")
    print(f"Portfolio {p}")
    print(f"{'='*50}")

    d = load_daily(p)
    iv = load_interval(p)

    daily_models = train_daily_models(d)
    aug = build_august_daily_grid(d, daily_models, bias_mult=1.10)

    print("August daily predictions:")
    print(aug[['Date', 'pred_cv', 'pred_cct', 'pred_abd']].to_string(index=False))

    profile_cv, profile_cct, profile_abd = build_profiles(iv)

    rows = []
    for _, day_row in aug.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])

        cv_preds = []
        for iod in range(48):
            cv_share = profile_cv.get((dow, iod), 1/48)
            cv_preds.append(max(0, cv_share * day_row['pred_cv']))

        cv_preds = np.array(cv_preds)
        if cv_preds.sum() > 0:
            cv_preds = cv_preds * (day_row['pred_cv'] / cv_preds.sum())

        for iod in range(48):
            h = iod // 2
            mn = (iod % 2) * 30

            cv_pred = float(max(0, cv_preds[iod]))
            cct_pred = float(max(0, profile_cct.get((dow, iod), day_row['pred_cct'])))
            abd_pred = float(np.clip(profile_abd.get((dow, iod), day_row['pred_abd']), 0, 1))

            rows.append({
                'Month': 'August',
                'Day': dom,
                'Interval': f"{h}:{mn:02d}",
                f'Calls_Offered_{p}': round(cv_pred, 2),
                f'Abandoned_Calls_{p}': round(cv_pred * abd_pred, 2),
                f'Abandoned_Rate_{p}': round(abd_pred, 6),
                f'CCT_{p}': round(cct_pred, 2),
            })

    all_forecasts.append(pd.DataFrame(rows))
    print(f"{p}: {len(rows)} rows generated")

# ══════════════════════════════════════════════════════════════════════════════
# COMBINE
# ══════════════════════════════════════════════════════════════════════════════
print("\nCombining portfolios...")
result = all_forecasts[0][['Month', 'Day', 'Interval',
    'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A']]

for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(
        all_forecasts[i + 1][[
            'Month', 'Day', 'Interval',
            f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
            f'Abandoned_Rate_{p}', f'CCT_{p}'
        ]],
        on=['Month', 'Day', 'Interval'], how='left'
    )

col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]
result = result[col_order]

for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}'] = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}'] = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}'] = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Saved to {OUTPUT_PATH}")
print(f"   Shape:     {result.shape}  |  Expected: (1488, 19)")
print(f"   Negatives: {(result.select_dtypes('number') < 0).any().any()}")
print(f"   Nulls:     {result.isnull().any().any()}")

print("\n=== Daily CV totals by portfolio ===")
for p in PORTFOLIOS:
    col = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")

print("\n=== First 5 rows ===")
print(result.head(5).to_string(index=False))


Portfolio A
  A daily nulls after cleanup: 0
  A interval nulls after cleanup: 0
Daily training rows: 731
August daily predictions:
      Date     pred_cv   pred_cct  pred_abd
2026-08-01 3066.140838 319.714737  0.010738
2026-08-02 1797.789362 315.483767  0.009880
2026-08-03 4878.666103 299.660955  0.011386
2026-08-04 4241.223407 309.239447  0.014857
2026-08-05 4605.752658 323.501907  0.012200
2026-08-06 4730.927551 308.706836  0.012819
2026-08-07 4621.612423 314.500423  0.010038
2026-08-08 3446.420729 303.999392  0.009790
2026-08-09 1670.662033 314.968798  0.008580
2026-08-10 4807.349603 297.678073  0.009067
2026-08-11 4520.414692 313.547682  0.010518
2026-08-12 4203.512397 316.403895  0.007882
2026-08-13 4263.217204 314.392708  0.011667
2026-08-14 4181.830148 320.211202  0.013868
2026-08-15 3222.328090 310.856885  0.012603
2026-08-16 1471.639549 323.359911  0.011969
2026-08-17 4387.813788 309.731487  0.013649
2026-08-18 4319.004076 321.717913  0.009359
2026-08-19 4372.882816 325.1776

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

PORTFOLIOS  = ['A', 'B', 'C', 'D']
DATA_DIR    = './data'
OUTPUT_PATH = './forecast_v08.csv'

INTERVAL_FEATURES = [
    'IntervalIdx', 'day_of_week', 'day_of_month', 'month', 'week_of_month',
    'is_weekend', 'is_holiday', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'daily_cv', 'load_factor',
    'cv_lag_1', 'cv_lag_2', 'cv_lag_48', 'cv_lag_96',
    'cct_lag_1', 'cct_lag_2', 'cct_lag_48', 'cct_lag_96',
    'abd_lag_1', 'abd_lag_2', 'abd_lag_48', 'abd_lag_96',
]

def load_interval(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Interval.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Interval'] = df['Interval'].apply(
        lambda x: x.strftime('%H:%M:%S') if hasattr(x, 'strftime') else str(x)
    )
    dr = pd.date_range('2025-04-01', '2025-06-30 23:30', freq='30min')
    skeleton = pd.DataFrame({'Month': dr.month_name(), 'Day': dr.day, 'Interval': dr.strftime('%H:%M:%S')})
    df = skeleton.merge(df, on=['Month', 'Day', 'Interval'], how='left')
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df.groupby(['Month', 'Day'])[col].transform(
            lambda x: x.interpolate(method='linear').bfill().ffill()
        ).clip(lower=0)
    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)
    df['Date']         = pd.to_datetime(df['Month'] + ' ' + df['Day'].astype(str) + ' 2025')
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['day_of_month'] = df['Date'].dt.day
    df['month']        = df['Date'].dt.month
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    df['is_holiday']   = df['Date'].isin(pd.to_datetime(['2025-05-26', '2025-06-19'])).astype(int)
    df['IntervalIdx']  = df['Interval'].apply(
        lambda x: int(x.split(':')[0]) * 2 + int(x.split(':')[1]) // 30
    )
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df.groupby(['day_of_week', 'IntervalIdx'])[col].transform(
            lambda x: x.fillna(x.median())
        )
    for col in ['Call Volume', 'CCT', 'Abandoned Calls', 'Abandoned Rate']:
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)
    df['Abandoned Rate'] = df['Abandoned Rate'].clip(0, 1)
    df['hour_sin'] = np.sin(2 * np.pi * df['IntervalIdx'] / 48)
    df['hour_cos'] = np.cos(2 * np.pi * df['IntervalIdx'] / 48)
    df['dow_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df = df.sort_values(['Date', 'IntervalIdx']).reset_index(drop=True)
    for lag in [1, 2, 48, 96]:
        df[f'cv_lag_{lag}']  = df['Call Volume'].shift(lag)
        df[f'cct_lag_{lag}'] = df['CCT'].shift(lag)
        df[f'abd_lag_{lag}'] = df['Abandoned Rate'].shift(lag)
    return df

def load_daily(p):
    df = pd.read_csv(f'{DATA_DIR}/{p} - Daily.csv', index_col=0)
    df.columns = df.columns.str.strip()
    df['Date']         = pd.to_datetime(df['Date'].str[:8], format='%m/%d/%y')
    df                 = df.sort_values('Date').reset_index(drop=True)
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['month']        = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_month']= (df['day_of_month'] - 1) // 7 + 1
    df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('day_of_week')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df.groupby('month')[col].transform(lambda x: x.fillna(x.median()))
    for col in ['Call Volume', 'CCT', 'Abandon Rate']:
        df[col] = df[col].fillna(df[col].median()).clip(lower=0)
    return df

aug_days      = pd.date_range('2026-08-01', '2026-08-31', freq='D')
all_forecasts = []

for p in PORTFOLIOS:
    print(f"\n{'='*50}\nPortfolio {p}\n{'='*50}")

    d  = load_daily(p)
    iv = load_interval(p)
    iv = iv.merge(
        d[['Date', 'Call Volume']].rename(columns={'Call Volume': 'daily_cv'}),
        on='Date', how='left'
    )
    iv['load_factor'] = iv['Call Volume'] / iv['daily_cv'].clip(lower=1)

    # August anchor from 2024+2025 actuals
    aug_actual     = d[d['month'] == 8].copy()
    aug_dow_cv     = aug_actual.groupby('day_of_week')['Call Volume'].mean()
    aug_dow_cct    = aug_actual.groupby('day_of_week')['CCT'].mean()
    aug_dow_abd    = aug_actual.groupby('day_of_week')['Abandon Rate'].mean()
    aug_dom_factor = aug_actual.groupby('day_of_month')['Call Volume'].mean()
    aug_dom_factor = aug_dom_factor / aug_dom_factor.mean()

    aug = pd.DataFrame({
        'Date':          aug_days,
        'day_of_week':   aug_days.dayofweek,
        'day_of_month':  aug_days.day,
        'week_of_month': (aug_days.day - 1) // 7 + 1,
        'is_weekend':    (aug_days.dayofweek >= 5).astype(int),
        'month': 8, 'is_holiday': 0,
    })
    aug['pred_cv']  = (
        aug['day_of_week'].map(aug_dow_cv) *
        aug['day_of_month'].map(aug_dom_factor).fillna(1.0) * 1.10
    ).clip(lower=0)
    aug['pred_cct'] = aug['day_of_week'].map(aug_dow_cct).clip(lower=0)
    aug['pred_abd'] = aug['day_of_week'].map(aug_dow_abd).clip(lower=0)

    # Intraday profiles — June weighted 2x
    iv_weighted = pd.concat([iv, iv[iv['month'] == 6]], ignore_index=True)
    iv_weighted['cv_share'] = iv_weighted['Call Volume'] / iv_weighted['daily_cv'].clip(lower=1)
    profile_cv   = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['cv_share'].mean()
    dow_sums     = profile_cv.groupby(level='day_of_week').transform('sum')
    profile_cv   = (profile_cv / dow_sums.clip(lower=1e-9)).to_dict()
    profile_cct  = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['CCT'].mean().to_dict()
    profile_abd  = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['Abandoned Rate'].mean().to_dict()
    profile_load = iv_weighted.groupby(['day_of_week', 'IntervalIdx'])['load_factor'].mean().to_dict()

    # Train interval models with lag features
    train_iv = iv[iv['month'].isin([4, 5])].dropna(
        subset=INTERVAL_FEATURES + ['Call Volume', 'CCT', 'Abandoned Rate']
    )
    val_iv = iv[iv['month'] == 6].dropna(subset=INTERVAL_FEATURES + ['Call Volume'])
    print(f"Train: {len(train_iv)} | Val: {len(val_iv)}")

    iv_models = {}
    for target in ['Call Volume', 'CCT', 'Abandoned Rate']:
        m = HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.05, max_depth=6,
            min_samples_leaf=20, random_state=42
        )
        m.fit(train_iv[INTERVAL_FEATURES], train_iv[target])
        preds = np.maximum(0, m.predict(val_iv[INTERVAL_FEATURES]))
        mask  = val_iv[target] > 0
        if mask.sum() > 0:
            mape = mean_absolute_percentage_error(val_iv[target][mask], preds[mask]) * 100
            print(f"  {target}: MAPE={mape:.2f}%")
        iv_models[target] = m

    # Build August grid vectorised
    grid_rows = []
    for _, day_row in aug.iterrows():
        dow = int(day_row['day_of_week'])
        dom = int(day_row['day_of_month'])
        wom = int((dom - 1) // 7 + 1)
        daily_cv_val = day_row['pred_cv']

        for iod in range(48):
            iod_lag1 = (iod - 1) % 48
            iod_lag2 = (iod - 2) % 48
            cv_p  = profile_cv.get((dow, iod), 1/48) * daily_cv_val
            cct_p = profile_cct.get((dow, iod), day_row['pred_cct'])
            abd_p = profile_abd.get((dow, iod), 0.01)
            grid_rows.append({
                'IntervalIdx': iod, 'day_of_week': dow, 'day_of_month': dom,
                'month': 8, 'week_of_month': wom, 'is_weekend': int(dow >= 5), 'is_holiday': 0,
                'hour_sin': np.sin(2*np.pi*iod/48), 'hour_cos': np.cos(2*np.pi*iod/48),
                'dow_sin': np.sin(2*np.pi*dow/7), 'dow_cos': np.cos(2*np.pi*dow/7),
                'daily_cv': daily_cv_val,
                'load_factor': profile_load.get((dow, iod), 1/48),
                'cv_lag_1':  profile_cv.get((dow, iod_lag1), 1/48) * daily_cv_val,
                'cv_lag_2':  profile_cv.get((dow, iod_lag2), 1/48) * daily_cv_val,
                'cv_lag_48': cv_p, 'cv_lag_96': cv_p,
                'cct_lag_1': profile_cct.get((dow, iod_lag1), day_row['pred_cct']),
                'cct_lag_2': profile_cct.get((dow, iod_lag2), day_row['pred_cct']),
                'cct_lag_48': cct_p, 'cct_lag_96': cct_p,
                'abd_lag_1': profile_abd.get((dow, iod_lag1), 0.01),
                'abd_lag_2': profile_abd.get((dow, iod_lag2), 0.01),
                'abd_lag_48': abd_p, 'abd_lag_96': abd_p,
                '_dow': dow, '_iod': iod, '_dom': dom,
                '_cv_p': cv_p, '_cct_p': cct_p, '_abd_p': abd_p,
            })

    aug_grid = pd.DataFrame(grid_rows)

    cv_model  = np.maximum(0, iv_models['Call Volume'].predict(aug_grid[INTERVAL_FEATURES]))
    cct_model = np.maximum(0, iv_models['CCT'].predict(aug_grid[INTERVAL_FEATURES]))
    abd_model = np.clip(iv_models['Abandoned Rate'].predict(aug_grid[INTERVAL_FEATURES]), 0, 1)

    cv_final  = (0.7 * cv_model  + 0.3 * aug_grid['_cv_p'].values).clip(min=0)
    cct_final = (0.7 * cct_model + 0.3 * aug_grid['_cct_p'].values).clip(min=0)
    abd_final = np.clip(0.7 * abd_model + 0.3 * aug_grid['_abd_p'].values, 0, 1)

    aug_grid['Month']    = 'August'
    aug_grid['Day']      = aug_grid['_dom']
    aug_grid['Interval'] = aug_grid['_iod'].apply(lambda x: f"{x//2}:{(x%2)*30:02d}")
    aug_grid[f'Calls_Offered_{p}']   = cv_final.round(2)
    aug_grid[f'Abandoned_Calls_{p}'] = (cv_final * abd_final).round(2)
    aug_grid[f'Abandoned_Rate_{p}']  = abd_final.round(6)
    aug_grid[f'CCT_{p}']             = cct_final.round(2)

    all_forecasts.append(aug_grid[[
        'Month', 'Day', 'Interval',
        f'Calls_Offered_{p}', f'Abandoned_Calls_{p}',
        f'Abandoned_Rate_{p}', f'CCT_{p}'
    ]])
    print(f"{p}: {len(aug_grid)} rows generated")

# Combine
print("\nCombining portfolios...")
result = all_forecasts[0]
for i, p in enumerate(['B', 'C', 'D']):
    result = result.merge(all_forecasts[i + 1], on=['Month', 'Day', 'Interval'], how='left')

col_order = ['Month', 'Day', 'Interval']
for p in PORTFOLIOS:
    col_order += [f'Calls_Offered_{p}', f'Abandoned_Calls_{p}', f'Abandoned_Rate_{p}', f'CCT_{p}']
result = result[col_order]
for p in PORTFOLIOS:
    result[f'Calls_Offered_{p}']   = result[f'Calls_Offered_{p}'].clip(lower=0)
    result[f'Abandoned_Calls_{p}'] = result[f'Abandoned_Calls_{p}'].clip(lower=0)
    result[f'Abandoned_Rate_{p}']  = result[f'Abandoned_Rate_{p}'].clip(0, 1)
    result[f'CCT_{p}']             = result[f'CCT_{p}'].clip(lower=0)

result.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Saved to {OUTPUT_PATH}")
print(f"   Shape: {result.shape} | Negatives: {(result.select_dtypes('number')<0).any().any()} | Nulls: {result.isnull().any().any()}")
print("\n=== Daily CV totals ===")
for p in PORTFOLIOS:
    col = f'Calls_Offered_{p}'
    daily = result.groupby('Day')[col].sum()
    print(f"  {p}: mean={daily.mean():.0f}  min={daily.min():.0f}  max={daily.max():.0f}")
print("\n=== First 5 rows ===")
print(result.head(5).to_string())


Portfolio A
Train: 2832 | Val: 1440
  Call Volume: MAPE=7.14%
  CCT: MAPE=24.95%
  Abandoned Rate: MAPE=56.31%
A: 1488 rows generated

Portfolio B
Train: 2832 | Val: 1440
  Call Volume: MAPE=6.65%
  CCT: MAPE=21.65%
  Abandoned Rate: MAPE=82.99%
B: 1488 rows generated

Portfolio C
Train: 2832 | Val: 1440
  Call Volume: MAPE=4.90%
  CCT: MAPE=9.42%
  Abandoned Rate: MAPE=131.42%
C: 1488 rows generated

Portfolio D
Train: 2832 | Val: 1440
  Call Volume: MAPE=6.97%
  CCT: MAPE=15.87%
  Abandoned Rate: MAPE=107.38%
D: 1488 rows generated

Combining portfolios...

✅ Saved to ./forecast_v08.csv
   Shape: (1488, 19) | Negatives: False | Nulls: False

=== Daily CV totals ===
  A: mean=4048  min=1475  max=5910
  B: mean=9158  min=4533  max=12576
  C: mean=20185  min=9600  max=27806
  D: mean=10638  min=5260  max=14615

=== First 5 rows ===
    Month  Day Interval  Calls_Offered_A  Abandoned_Calls_A  Abandoned_Rate_A   CCT_A  Calls_Offered_B  Abandoned_Calls_B  Abandoned_Rate_B   CCT_B  Calls_O